# Corrected notebook (15 November 2026): how to run it

**On Google Colab (recommended, needs a GPU):** File > Upload notebook, then Runtime > Change runtime type > T4 GPU, then Runtime > Run all. If the first cell restarts the runtime, press Run all once more. After each stage the results are zipped and downloaded by your browser into the **Downloads** folder of your computer (`mof_wulff_results_<stage>_<date>.zip`); if the browser asks whether to allow several downloads, allow them. Keep the tab open: the full run (16 MOFs relaxed, 41 static, the GNN) takes several hours on a T4.

**On any computer with Jupyter and Python 3.10+ (slow without a GPU):** open the notebook and run all; the zips are copied to `~/Downloads`.

Send the zip called `..._final_...` back; it holds every result. The earlier zips are checkpoints in case the session stops.

**What changed from the team version of 14 November** (details: `code/morphology/steps1_8_check/nb_2026-11-14/README.md` and the website page *ML energies and water*):

- **Water condition.** Surface energies are reported at the water chemical potential `WATER_DMU_EV` (cell 4; default liquid water at 25 °C). The old reference, a free water molecule at 0 K, gave six negative surface energies. The stored `gamma_*_J_m2` columns keep the old reference so old and new runs can be compared; the shapes and the GNN labels use `WATER_DMU_EV`.
- **Relaxed cell used consistently.** Slabs and bulk come from the same relaxed cell, and faces are converted to the right planes after the cell is reloaded (two fixes from 10 November that were missing).
- **Water caps.** A linker oxygen gets at most one hydrogen; a surplus hydrogen turns the nearest metal–OH cap into a water molecule on the metal (before, a bridging oxygen could get two).
- **Metal rods.** Metals that share a linker oxygen form one inorganic unit, so the Mg–O rod of Mg-MOF-74 is kept whole.
- **Shape names** come from the exact limits of the cubic shape map.
- **UiO-66 reference shape** is a cube without additives (octahedra need an acid modulator), and the UiO-66 input now has the Zr6O4(OH)4 node (the RASPA2 file lacks the hydroxide groups); it is embedded in cell 3.
- **Face ranking** uses the effective spacing d/n, so faces forbidden by centring or screw axes are no longer kept ahead of allowed ones.
- **An open Wulff shape** (a missing face family) is reported as open, not as a rod.
- **MACE check:** the rank correlation is also computed over all slabs of a MOF, not only over three faces.
- **DFT inputs:** slabs are turned so the normal is along z and get the surface dipole correction; open-shell metals get a spin multiplicity (Cu2+: one unpaired electron each, ferromagnetic; stated in the input).
- **Slab thickness check** (cell 5b): ZIF-8 {110} at 15, 22 and 30 A.
- **GNN.** Labels at `WATER_DMU_EV`; labels ≤ 0 are dropped and listed.
- Runs on Colab or locally; results go to Downloads; Google Drive is optional.


# MOF crystal habit from CHGNet surface energies: pipeline steps 1–8

Runs entirely in Colab: CIFs are downloaded (RASPA2 structure library on GitHub, or CoRE MOF 2019 on Zenodo by CSD refcode), models are downloaded by pip.

| Step | What runs here |
|---|---|
| 1 Standardize | conventional + working cell; disorder policy (`majority` / `reject`), overlapping split sites merged, free guests removed, formula / lattice / space group stored. If H or OH ordering breaks the symmetry, the point group comes from the H-free or metal+C skeleton (logged). |
| 2 Building blocks | one bond rule set for all steps; nodes (metal + oxo/OH + M–M) vs linkers; unwrapped coordinates and centroids; infinite rods/sheets detected and flagged |
| 3 Faces + terminations | symmetry-unique {hkl} families from the real point group (h' = hR⁻¹, Laue class), ranked by d_hkl (BFDH), not a fixed cubic list; terminations = distinct block-centroid levels, duplicates removed; if more than `MAX_TERMS`, the ones breaking fewest bonds per area are kept |
| 4 Slabs | whole blocks only; stoichiometry **asserted** (atom count = n × OUC, same reduced formula); vacuum. Neutral by construction (whole blocks, no charge transfer) |
| 5 Healing | **on by default**. Every broken bond is found by comparing each slab atom with its bulk image; each broken bond gets one H₂O, split heterolytically (OH⁻ on the metal, H⁺ on the donor O/N). Terminations that break organic or M–M bonds, or are unbalanced, are rejected. Intact slabs are kept (old bug fixed). |
| 6 Surface energies | CHGNet; bulk relaxed at fixed cell with the same settings; all terminations screened statically, the `RELAX_TOP_K` lowest relaxed (fmax and step limit logged); γ = (E_slab − N·e_bulk − n·E_H₂O)/2A; lowest converged γ per family; unconverged slabs excluded and listed |
| 7 Validation | (a) independent MLIP (MACE-MP-0) re-scores the CHGNet geometries: rank correlation and MAE; (b) CP2K PBE-D3(BJ) inputs for a small held-out set; (c) upload DFT energies → MAE / Spearman / failures. **DFT itself cannot run in Colab** for 300–600-atom slabs. 7(a) is a cross-check, not DFT. |
| 8 Wulff | Wulff polyhedron built from the true point group (pymatgen's WulffShape uses lattice symmetry, wrong for non-cubic MOFs); area fractions; habit name (cubic polyhedra, or rod / plate / prism for others); open (unbounded) directions reported; 3D PNG and OBJ per MOF |

**Runtime (Colab T4 GPU):** roughly 15–40 min per MOF with full relaxation. Fill `ONLY` with one MOF name to test first. Results are cached, so a rerun only computes what is missing. Turn on `SAVE_TO_DRIVE` to survive disconnects.

In [ ]:
# 1. Install (about 2 min). Restarts the runtime once if numpy was upgraded; then use Runtime > Run all again.
import importlib.util, subprocess, sys, os, shutil, time
BASE = os.getcwd()          # /content on Colab; the notebook's folder in local Jupyter
IN_COLAB = importlib.util.find_spec("google.colab") is not None

def deliver(tag):
    """Zip everything in OUT_DIR and put the zip in the Downloads folder of this computer.
    Colab: the browser downloads it (allow multiple downloads if the browser asks). Local: copied to ~/Downloads."""
    name = f"mof_wulff_results_{tag}_{time.strftime('%Y%m%d_%H%M')}"
    z = shutil.make_archive(os.path.join(BASE, name), "zip", OUT_DIR)
    if IN_COLAB:
        try:
            from google.colab import files; files.download(z); print("downloading", os.path.basename(z), "to your Downloads folder")
        except Exception as e:
            print("could not start the download:", e, "| the zip is at", z)
    else:
        dl = os.path.join(os.path.expanduser("~"), "Downloads"); os.makedirs(dl, exist_ok=True)
        shutil.copy(z, dl); print("saved", os.path.join(dl, os.path.basename(z)))
if importlib.util.find_spec("chgnet") is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "chgnet>=0.3.0", "pymatgen", "networkx"], check=True)
    import importlib.metadata as md
    if "numpy" in sys.modules and sys.modules["numpy"].__version__ != md.version("numpy"):
        print("numpy was upgraded: restarting runtime. Run all cells again."); os.kill(os.getpid(), 9)
import torch, chgnet, pymatgen.core
print("chgnet", chgnet.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("No GPU: Runtime > Change runtime type > T4 GPU (CPU works but is ~10x slower).")

In [ ]:
# 2. Output folder. Results are also zipped into your Downloads folder after each stage (deliver()).
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
OUT_DIR = os.path.join(BASE, "results")
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/mof_wulff_results"
os.makedirs(OUT_DIR, exist_ok=True); print("results ->", OUT_DIR)


In [ ]:
%%writefile mofwulff.py
"""MOF crystal habit from machine-learned surface energies (pipeline steps 1-8).

Step 1  standardize()        read CIF, disorder/overlap/guest policy, symmetry, conv + working cell
Step 2  building_blocks()    metal nodes vs organic linkers, unwrapped, periodic-node detection
Step 3  face_families()      symmetry-unique {hkl} families + distinct block terminations
Step 4  block_slabs()        whole-block, stoichiometric slabs (asserted), vacuum
Step 5  heal()               heterolytic H2O capping of every broken metal-ligand bond (H+ to ligand, OH- to metal)
Step 6  CHGNet energies      bulk relaxed at fixed cell; static screen of all terminations, relax lowest K
Step 7  validation helpers   independent-model (MACE-MP-0) re-scoring, CP2K PBE-D3 inputs, DFT comparison
Step 8  wulff()              Wulff polyhedron from true point-group symmetry, area fractions, habit name, 3D render

Every rule that discards or caps something is written in a docstring and logged in the outputs.
"""
import hashlib, itertools, json, math, re, time, warnings
from collections import Counter
from math import gcd
from pathlib import Path

import networkx as nx
import numpy as np
from pymatgen.core import Element, Lattice, Molecule, Structure
from pymatgen.core.surface import SlabGenerator
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

warnings.filterwarnings("ignore")
EV_A2_TO_J_M2 = 16.0218

# ----------------------------------------------------------------------------- bonding rules
# One set of rules, used by every step (blocks, slabs, healing, coordination counts).
ORGANIC = {("C", "C"): 1.65, ("C", "O"): 1.50, ("C", "H"): 1.20, ("C", "N"): 1.60, ("N", "H"): 1.15,
           ("O", "H"): 1.15, ("N", "N"): 1.50, ("C", "S"): 1.90, ("C", "F"): 1.45, ("C", "Cl"): 1.85,
           ("C", "Br"): 2.00, ("B", "O"): 1.60, ("B", "C"): 1.70, ("P", "O"): 1.70, ("S", "O"): 1.60}
DONORS = {"O", "N", "S", "F", "Cl"}       # atoms that can bind a metal
M_X = 2.55                                # metal-donor bond cutoff, A
M_M = 2.80                                # metal-metal bond cutoff, A (Cu2 paddlewheel 2.6; Mg...Mg in MOF-74 rods ~2.9 is not a bond)
OUT_REACH = 3.0                           # neighbour search radius, A


def is_metal(el):
    e = Element(el)
    return e.is_metal or el in {"Ge", "Sb"}  # Element.is_metal covers alkali..post-transition


def _org(a, b):
    return ORGANIC.get((a, b), ORGANIC.get((b, a)))


def bond_kind(a, b, d, lig_a=False, lig_b=False):
    """Classify a pair: 'org', 'MM', 'ML' (metal-ligand donor), 'MO' (metal-oxo/hydroxo/aqua donor) or None."""
    ma, mb = is_metal(a), is_metal(b)
    if ma and mb:
        return "MM" if d <= M_M else None
    if ma or mb:
        x, lx = (b, lig_b) if ma else (a, lig_a)
        if x in DONORS and d <= M_X:
            return "ML" if lx else "MO"
        return None
    r = _org(a, b)
    return "org" if r is not None and d <= r else None


def ligand_donors(s, nbs=None):
    """Indices of donor atoms bonded to C (carboxylate O, imidazolate N, ...): they belong to linkers."""
    nbs = nbs if nbs is not None else s.get_all_neighbors(OUT_REACH)
    sym = [x.specie.symbol for x in s]
    return {i for i in range(len(s)) if sym[i] in DONORS
            and any(n.specie.symbol == "C" and n.nn_distance <= _org(sym[i], "C") for n in nbs[i])}


def bond_counts(s, lig, nbs=None):
    """Per-site Counter of bond kinds. lig = set of ligand donor indices of s."""
    nbs = nbs if nbs is not None else s.get_all_neighbors(OUT_REACH)
    sym = [x.specie.symbol for x in s]
    out = []
    for i in range(len(s)):
        c = Counter()
        for n in nbs[i]:
            k = bond_kind(sym[i], sym[n.index], n.nn_distance, i in lig, n.index in lig)
            if k:
                c[k] += 1
        out.append(c)
    return out


# ----------------------------------------------------------------------------- step 1
def standardize(cif, symprec=0.1, disorder="majority", remove_guests=True, overlap_tol=0.6):
    """Read and clean a CIF.

    Disorder policy (documented, logged in notes):
      'reject'   : any partial occupancy -> error.
      'majority' : each disordered site keeps its highest-occupancy species; sites with total
                   occupancy < 0.5 are removed. Pairs of sites closer than overlap_tol (split
                   positions written as full occupancy) are merged, keeping the first.
    Guests: connected components with no metal and no bond to the framework (free solvent)
            are removed when remove_guests is True.
    Symmetry: tried on (a) full structure, (b) without H, (c) metals + C only. The highest
            space group found defines the point group used for face families. If (a) has it,
            conventional + primitive standard cells are used (slabs from the primitive cell).
            If only a skeleton has it (ordered H / ordered OH breaking symmetry in the CIF),
            the input cell is used as both cells, with the skeleton's operations.
    """
    notes = []
    s = Structure.from_file(cif)
    if not s.is_ordered:
        n_dis = sum(1 for site in s if not site.is_ordered)
        if disorder == "reject":
            raise ValueError(f"{n_dis} partially occupied sites; disorder policy is 'reject'")
        sp, fc, drop = [], [], 0
        for site in s:
            occ = site.species.num_atoms
            if occ < 0.5:
                drop += 1
                continue
            el = max(site.species.items(), key=lambda kv: kv[1])[0]
            sp.append(el.symbol if hasattr(el, "symbol") else str(el)); fc.append(site.frac_coords)
        s = Structure(s.lattice, sp, fc)
        notes.append(f"disorder: {n_dis} partial sites -> majority species, {drop} low-occupancy sites removed")
    # overlapping split sites
    dm = s.distance_matrix
    np.fill_diagonal(dm, 99)
    close = sorted({max(i, j) for i, j in zip(*np.where(dm < overlap_tol))})
    if close:
        s.remove_sites(close)
        notes.append(f"overlap: {len(close)} sites closer than {overlap_tol} A to another site removed")
    # free guests
    if remove_guests:
        nbs = s.get_all_neighbors(OUT_REACH)
        lig = ligand_donors(s, nbs)
        sym = [x.specie.symbol for x in s]
        G = nx.Graph(); G.add_nodes_from(range(len(s)))
        for i in range(len(s)):
            for n in nbs[i]:
                if bond_kind(sym[i], sym[n.index], n.nn_distance, i in lig, n.index in lig):
                    G.add_edge(i, n.index)
        guests = [c for c in nx.connected_components(G) if not any(is_metal(sym[i]) for i in c)]
        if guests:
            idx = sorted(i for c in guests for i in c)
            formulas = Counter(Structure(s.lattice, [sym[i] for i in c], [s[i].frac_coords for i in c])
                               .composition.reduced_formula for c in guests)
            s.remove_sites(idx)
            notes.append(f"guests removed: {dict(formulas)}")
    # symmetry
    def sg(st):
        try:
            a = SpacegroupAnalyzer(st, symprec=symprec)
            return a.get_space_group_number(), a
        except Exception:
            return 1, None
    full_n, full_a = sg(s)
    noH = s.copy(); noH.remove_species(["H"])
    mc = s.copy(); mc.remove_species([e for e in {x.specie.symbol for x in s} if not (is_metal(e) or e == "C")])
    cands = [("full", full_n, full_a, s), ("no_H", *sg(noH), noH), ("metal+C", *sg(mc), mc)]
    best = max(cands, key=lambda c: c[1])
    if best[0] == "full" or full_n == best[1]:
        conv = full_a.get_conventional_standard_structure()
        # primitive cell in the SAME cartesian frame as conv (needed for exact Miller conversion)
        prim = conv.get_primitive_structure(tolerance=0.25)
        if len(prim) != len(full_a.get_primitive_standard_structure()):
            prim = conv
            notes.append("primitive reduction in the conventional frame failed; conventional cell used for slabs")
        ops = SpacegroupAnalyzer(conv, symprec=symprec).get_symmetry_operations()
        sym_src, cs, sgsym = "full", full_a.get_crystal_system(), full_a.get_space_group_symbol()
    else:
        conv = prim = s
        ops = best[2].get_symmetry_operations()
        sym_src, cs, sgsym = best[0], best[2].get_crystal_system(), best[2].get_space_group_symbol()
        notes.append(f"symmetry: full structure is SG {full_n}; skeleton '{best[0]}' is {sgsym}; "
                     f"input cell used, skeleton point group used for face families")
    rots = []
    for op in ops:
        R = np.round(op.rotation_matrix).astype(int)
        if not any(np.array_equal(R, q) for q in rots):
            rots.append(R)
    # full operations (R, t) in the basis of conv, for the systematic-absence test of face_families (K7)
    sym_ops = [(np.round(op.rotation_matrix).astype(int), np.array(op.translation_vector, float)) for op in ops]
    return dict(structure=s, conv=conv, prim=prim, rotations=rots, ops=sym_ops, space_group=sgsym,
                symmetry_source=sym_src, crystal_system=cs, formula=s.composition.reduced_formula,
                n_atoms_conv=len(conv), n_atoms_prim=len(prim), notes=notes)


# ----------------------------------------------------------------------------- step 2
def building_blocks(s, join_shared_donors=True):
    """Split a periodic structure into building blocks.

    node   = metal atoms + donors bonded only to metals (oxo/hydroxo/aqua), joined by M-M and M-oxo bonds
    linker = connected organic piece (ORGANIC bonds), including its donor atoms
    Only 'ML' bonds (metal - ligand donor) are ever cut between blocks.
    A block that wraps onto itself through a lattice translation is an infinite (rod/sheet)
    inorganic unit; it is cut at the cell boundary and flagged (periodic_dirs, cartesian).
    """
    nbs = s.get_all_neighbors(OUT_REACH)
    lig = ligand_donors(s, nbs)
    sym = [x.specie.symbol for x in s]
    G = nx.MultiGraph(); G.add_nodes_from(range(len(s)))
    for i in range(len(s)):
        for n in nbs[i]:
            k = bond_kind(sym[i], sym[n.index], n.nn_distance, i in lig, n.index in lig)
            if k and k != "ML" and i <= n.index:
                G.add_edge(i, n.index, img=np.array(n.image, int))
    if join_shared_donors:
        # K2 (15 Nov 2026): two metals bonded to the same linker donor (a bridging phenolate or carboxylate O,
        # as along the Mg-O rod of Mg-MOF-74) belong to one inorganic unit. They are joined by a virtual
        # metal-metal edge; the donor stays in its linker, so only metal-donor bonds are still cut between blocks.
        for d in lig:
            ms = [(n.index, np.array(n.image, int)) for n in nbs[d]
                  if is_metal(n.specie.symbol) and bond_kind(sym[d], n.specie.symbol, n.nn_distance, True, False) == "ML"]
            for (a, ia), (b, ib) in itertools.combinations(ms, 2):
                if a == b and np.array_equal(ia, ib):
                    continue
                i, j, t = (a, b, ib - ia) if a <= b else (b, a, ia - ib)
                if i == j and not np.any(t):
                    continue
                G.add_edge(i, j, img=t)
    frac = s.frac_coords
    blocks = []
    for comp in nx.connected_components(G):
        comp = sorted(comp); root = comp[0]
        off = {root: np.zeros(3, int)}; stack = [root]
        while stack:
            u = stack.pop()
            for _, v, data in G.edges(u, data=True):
                img = data["img"] if u <= v else -data["img"]
                if u == v:
                    continue
                if v not in off:
                    off[v] = off[u] + img; stack.append(v)
        per = set()
        for u, v, data in G.subgraph(comp).edges(data=True):
            i, j = min(u, v), max(u, v)
            t = off[i] + data["img"] - off[j]
            if np.any(t):
                per.add(tuple(int(x) for x in t))
        f = np.array([frac[i] + off[i] for i in comp])
        cart = s.lattice.get_cartesian_coords(f)
        blocks.append(dict(atoms=comp, species=[sym[i] for i in comp], cart=cart, centroid=cart.mean(0),
                           kind="node" if any(is_metal(sym[i]) for i in comp) else "linker",
                           formula=Structure(s.lattice, [sym[i] for i in comp], f).composition.formula.replace(" ", ""),
                           periodic_dirs=[s.lattice.get_cartesian_coords(np.array(t)).tolist() for t in per]))
    return blocks, lig


def summarize_blocks(blocks):
    c = Counter((b["kind"], b["formula"]) for b in blocks)
    per = any(b["periodic_dirs"] for b in blocks)
    return {"blocks": {f"{k}:{f}": n for (k, f), n in c.items()}, "infinite_inorganic_unit": per}


# ----------------------------------------------------------------------------- step 3
def _reduce(v):
    v = np.array(v, int); g = gcd(gcd(abs(v[0]), abs(v[1])), abs(v[2]))
    return tuple(int(x) for x in v // g)


def face_families(conv, rotations, max_index=1, n_keep=6, faces=None, ops=None):
    """Symmetry-unique face families.

    Miller indices transform as h' = h R^-1 for each point operation R (fractional), plus
    inversion (h and -h are the two sides of one slab, so families follow the Laue class).
    All primitive hkl with |h|,|k|,|l| <= max_index are grouped; the n_keep families with the
    largest interplanar spacing d_hkl are kept (BFDH ranking; documented, not random).
    faces: optional explicit list of hkl, which overrides the enumeration.
    """
    Rinv = [np.round(np.linalg.inv(R)).astype(int) for R in rotations]
    def orbit(h):
        out = set()
        for Ri in Rinv:
            hp = tuple(int(x) for x in np.array(h) @ Ri)
            out.add(hp); out.add(tuple(-x for x in hp))
        return sorted(out)
    if faces:
        cand = [_reduce(h) for h in faces]
    else:
        rng = range(-max_index, max_index + 1)
        cand = [h for h in itertools.product(rng, rng, rng) if any(h) and _reduce(h) == h]
    fams, seen = [], set()
    for h in cand:
        if h in seen:
            continue
        orb = orbit(h); seen.update(orb)
        rep = max(orb)                      # canonical representative: lexicographically largest
        d = conv.lattice.d_hkl(rep)
        order = 1 if faces else absence_order(conv, rep, ops)
        fams.append(dict(rep=rep, members=orb, d_hkl=d, order=order, d_eff=d / order,
                         name="{" + "".join(map(str, rep)).replace("-", "̅") + "}",
                         key="".join(str(x) for x in rep).replace("-", "m")))
    # K7 (15 Nov 2026): rank by the effective spacing d_hkl / n (Donnay-Harker), where n is the first order
    # of the reflection that symmetry does not extinguish; ranking by d_hkl kept faces that centring or
    # screw axes forbid.
    fams.sort(key=lambda f: (-f["d_eff"], -f["d_hkl"]))
    return fams if faces else fams[:n_keep]


def absence_order(conv, hkl, ops=None, n_max=6, symprec=0.1):
    """Smallest n (1..n_max) for which the reflection n*(hkl) is not a systematic absence.
    Uses the space-group operations (R, t) of the cell (fractional, same basis as hkl): a reflection h is
    extinguished when some operation leaves it fixed (h R = h) while h.t is not an integer. This is exact
    and, unlike a structure factor of the file's atoms, not disturbed by small deviations of the atoms
    from their ideal positions."""
    if ops is None:
        ops = [(o.rotation_matrix, o.translation_vector)
               for o in SpacegroupAnalyzer(conv, symprec=symprec).get_symmetry_operations(cartesian=False)]
    for n in range(1, n_max + 1):
        h = n * np.array(hkl, float)
        extinct = any(np.allclose(h @ R, h, atol=1e-6) and abs((h @ t) - round(h @ t)) > 1e-3 for R, t in ops)
        if not extinct:
            return n
    return n_max


# ----------------------------------------------------------------------------- step 4
def primitive_miller(conv, prim, hkl):
    """Conventional (hkl) -> integer Miller indices of the same plane in the primitive cell."""
    if conv is prim:
        return tuple(hkl)
    g = np.array(hkl, float) @ conv.lattice.reciprocal_lattice_crystallographic.matrix
    hp = prim.lattice.matrix @ g
    hp = hp / np.abs(hp[np.abs(hp) > 1e-6]).min()
    for m in range(1, 25):
        v = hp * m
        if np.allclose(v, np.round(v), atol=1e-3):
            return _reduce(np.round(v).astype(int))
    raise ValueError(f"cannot convert {hkl} to the primitive cell")


def _surface_fingerprint(blocks_in_slab, z, top, bottom, depth=4.0):
    t = sorted(f for f, zc in blocks_in_slab if zc > top - depth)
    b = sorted(f for f, zc in blocks_in_slab if zc < bottom + depth)
    return tuple(sorted([tuple(t), tuple(b)]))


def block_slabs(conv, prim, hkl, min_thickness=15.0, vacuum=15.0, level_tol=0.3):
    """All distinct whole-block slabs for conventional (hkl).

    Oriented unit cell (OUC) of the working cell; h = lattice repeat along the normal.
    A block copy is kept if its centroid height lies in [z0, z0 + n*h); every block type repeats
    with period h, so the slab holds exactly n copies of the OUC -> stoichiometric for any z0
    (asserted below). Terminations: one z0 just below each distinct centroid level (levels closer
    than level_tol merged). Duplicate terminations (same block formulas within 4 A of both
    surfaces) are removed.
    Returns list of dicts with slab, orig (OUC index per slab atom), OUC, metadata.
    """
    # oriented unit cell from the primitive and from the conventional cell; the smaller one is used
    opts = [(conv, tuple(hkl))]
    try:
        opts.append((prim, primitive_miller(conv, prim, hkl)))
    except ValueError:
        pass
    cells = [(SlabGenerator(cell, h_, 10, 10, primitive=False).oriented_unit_cell, h_) for cell, h_ in opts]
    # K3 fix (10 Nov 2026): on a tie prefer the working (relaxed) cell, so slab and bulk energies come
    # from the same geometry.
    ouc, hp = min(cells[::-1], key=lambda x: len(x[0]))
    a, b, c = ouc.lattice.matrix
    nrm = np.cross(a, b); nrm /= np.linalg.norm(nrm)
    h = float(np.dot(c, nrm))
    if h < 0:
        c, h = -c, -h
    n_rep = max(1, int(np.ceil(min_thickness / h)))
    blocks, lig = building_blocks(ouc)
    cuts_rod = any(abs(np.dot(np.array(d) / np.linalg.norm(d), nrm)) > 0.05
                   for bl in blocks for d in bl["periodic_dirs"])
    heights = np.sort(np.array([np.dot(bl["centroid"], nrm) for bl in blocks]) % h)
    distinct = [heights[0]]
    for zz in heights[1:]:
        if zz - distinct[-1] > level_tol:
            distinct.append(zz)
    if len(distinct) > 1 and distinct[0] + h - distinct[-1] <= level_tol:
        distinct.pop()
    out, prints = [], set()
    for z0 in [zz - level_tol / 2 for zz in distinct]:
        species, coords, orig, inslab = [], [], [], []
        for bl in blocks:
            zc = float(np.dot(bl["centroid"], nrm))
            k_lo = int(np.ceil((z0 - zc) / h))
            for k in range(k_lo, k_lo + n_rep):
                species += bl["species"]; coords += list(bl["cart"] + k * c); orig += bl["atoms"]
                inslab.append((bl["formula"], zc + k * h))
        coords = np.array(coords); zs = coords @ nrm
        top, bot = zs.max(), zs.min()
        fp = _surface_fingerprint(inslab, zs, top, bot)
        if fp in prints:
            continue
        prints.add(fp)
        coords = coords - np.outer(np.full(len(coords), bot - vacuum / 2), nrm)
        lat = Lattice([a, b, nrm * (top - bot + vacuum)])
        st = Structure(lat, species, coords, coords_are_cartesian=True)
        # Step 4 check: stoichiometry
        if len(st) != n_rep * len(ouc) or st.composition.reduced_composition != ouc.composition.reduced_composition:
            raise AssertionError(f"slab {hkl} z0={z0:.2f} is not stoichiometric")
        out.append(dict(slab=st, orig=orig, ouc=ouc, ouc_lig=lig, z0=round(float(z0), 3), n_layers=n_rep,
                        n_bulk_units=n_rep, thickness_A=round(float(top - bot), 2), repeat_A=round(h, 3),
                        primitive_hkl=hp, cuts_infinite_unit=cuts_rod,
                        area_A2=float(np.linalg.norm(np.cross(a, b)))))
    return out


def broken_bonds(sl):
    """Compare every slab atom's bonds with its bulk (OUC) image. Returns per-atom missing Counter."""
    ouc, st, orig = sl["ouc"], sl["slab"], sl["orig"]
    bulk = bond_counts(ouc, sl["ouc_lig"])
    lig_slab = {i for i, o in enumerate(orig) if o in sl["ouc_lig"]}
    now = bond_counts(st, lig_slab)
    miss = []
    for i, o in enumerate(orig):
        m = bulk[o] - now[i]                          # Counter subtraction keeps positives
        extra = now[i] - bulk[o]
        if extra:
            m["_extra"] = sum(extra.values())         # new contacts across the cut (should be 0)
        miss.append(m)
    tot = Counter()
    for m in miss:
        tot.update(m)
    return miss, dict(tot)


# ----------------------------------------------------------------------------- step 5
def _fib_sphere(n=96):
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n); th = np.pi * (1 + 5 ** 0.5) * i
    return np.c_[np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)]


DIRS = _fib_sphere()


def _best_dirs(center, occupied, k, length, others, min_clear):
    """k directions from center, each as far as possible (angle) from occupied directions,
    with the new atom at least min_clear from all other atoms."""
    chosen = []
    occ = [o / np.linalg.norm(o) for o in occupied if np.linalg.norm(o) > 1e-6]
    for _ in range(k):
        best, bscore = None, -9
        for d in DIRS:
            cos = max([np.dot(d, o) for o in occ + chosen] or [-1])
            p = center + length * d
            clear = np.min(np.linalg.norm(others - p, axis=1)) if len(others) else 9
            score = -cos + (0 if clear >= min_clear else -5)
            if score > bscore:
                best, bscore = d, score
        chosen.append(best)
    return chosen


def heal(sl, miss, ML_only=True, one_proton_per_donor=True):
    """Heal broken metal-donor bonds with heterolytic water (documented chemistry):
         each missing bond on a metal          -> +OH  (O at 2.0 A, H at 0.97 A)
         each missing bond on a donor atom     -> +H   (0.98 A)
    n_water = number of broken bonds. Balanced iff metal-side and donor-side counts match.
    Slabs with broken organic or M-M bonds, or unbalanced counts, are rejected.

    one_proton_per_donor (added 15 Nov 2026): a donor atom that loses two or more metal bonds (a bridging
    phenolate or carboxylate O, as in MOF-74) used to receive one H per lost bond, i.e. an O bonded to C and
    two H (C-OH2+), which is not a stable group. Now it receives at most one H in total (counting any H it
    already had), and every surplus proton goes to the nearest metal-side OH cap, which becomes a water
    molecule bound to the metal (O at 2.1 A, H-O-H 104.5 degrees, both H pointing away from the metal).
    The atoms added, the charge balance and n_water are unchanged; only where the protons sit changes.
    Returns (structure or None, n_water, status)."""
    st = sl["slab"]
    tot = Counter()
    for m in miss:
        tot.update(m)
    if tot.get("org", 0) or tot.get("MM", 0):
        return None, 0, "rejected: organic or metal-metal bond broken"
    if tot.get("_extra", 0):
        return None, 0, "rejected: new contacts across the cut"
    m_side = sum(m.get("ML", 0) + m.get("MO", 0) for i, m in enumerate(miss) if is_metal(st[i].specie.symbol))
    x_side = sum(m.get("ML", 0) + m.get("MO", 0) for i, m in enumerate(miss) if not is_metal(st[i].specie.symbol))
    if m_side != x_side:
        return None, 0, f"rejected: unbalanced ({m_side} metal-side vs {x_side} donor-side)"
    if m_side == 0:
        return st.copy(), 0, "intact (nothing to heal)"
    species = [x.specie.symbol for x in st]
    coords = [x.coords.copy() for x in st]
    nbs = st.get_all_neighbors(2.6)
    caps = []                                    # metal-side caps: [metal index, unit direction, O index, H index]
    surplus = []                                 # donor atoms whose extra protons must move to a cap
    for i, m in enumerate(miss):
        k = m.get("ML", 0) + m.get("MO", 0)
        if not k:
            continue
        x = st[i].coords
        occ = [n.coords - x for n in nbs[i] if n.nn_distance < 2.6]
        others = np.array(coords)
        others = others[np.linalg.norm(others - x, axis=1) > 0.1]
        if is_metal(species[i]):
            for d in _best_dirs(x, occ, k, 2.0, others, 1.6):
                o = x + 2.0 * d; species += ["O", "H"]; coords += [o, o + 0.97 * d]; occ.append(d)
                caps.append([i, d, len(species) - 2, len(species) - 1])
        else:
            if one_proton_per_donor:
                have = sum(1 for n in nbs[i] if n.specie.symbol == "H" and n.nn_distance < 1.2)
                n_h = max(0, min(k, 1 - have))
                surplus += [i] * (k - n_h)
            else:
                n_h = k
            for d in _best_dirs(x, occ, n_h, 0.98, others, 1.3) if n_h else []:
                species.append("H"); coords.append(x + 0.98 * d); occ.append(d)
    # surplus protons: turn the nearest still-OH cap into a bound water (same atoms, same charge balance)
    free = list(range(len(caps)))
    half = math.radians(104.5) / 2
    for i in surplus:
        x = st[i].coords
        j = min(free, key=lambda c: np.linalg.norm(coords[caps[c][2]] - x))
        free.remove(j)
        mi, d, oi, hi = caps[j]
        o = st[mi].coords + 2.1 * d
        p0 = np.cross(d, [1.0, 0, 0]); p0 = p0 if np.linalg.norm(p0) > 0.1 else np.cross(d, [0, 1.0, 0])
        p0 /= np.linalg.norm(p0); q0 = np.cross(d, p0)
        rest = np.array([c for n, c in enumerate(coords) if n not in (oi, hi)])
        best = None                                  # orientation of the H-O-H plane: farthest from other atoms
        for a in np.linspace(0, np.pi, 12, endpoint=False):
            p = math.cos(a) * p0 + math.sin(a) * q0
            h1 = o + 0.97 * (math.cos(half) * d + math.sin(half) * p)
            h2 = o + 0.97 * (math.cos(half) * d - math.sin(half) * p)
            clear = min(np.min(np.linalg.norm(rest - h1, axis=1)), np.min(np.linalg.norm(rest - h2, axis=1)))
            if best is None or clear > best[0]:
                best = (clear, h1, h2)
        coords[oi] = o
        coords[hi] = best[1]
        species.append("H"); coords.append(best[2])
    status = "healed" + (f" ({len(surplus)} bound H2O)" if surplus else "")
    return Structure(st.lattice, species, coords, coords_are_cartesian=True), m_side, status


# ----------------------------------------------------------------------------- step 6
class Engine:
    """CHGNet wrapper: static energies (batched) and fixed-cell relaxations."""
    def __init__(self, device=None):
        import torch
        from chgnet.model import CHGNet
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CHGNet.load(use_device=self.device, verbose=False)

    def static(self, structs):
        # batches of 4 on a GPU; one at a time on a CPU, where a batch of large slabs can exhaust the memory
        preds = self.model.predict_structure(list(structs), task="e", batch_size=4 if self.device == "cuda" else 1)
        preds = preds if isinstance(preds, list) else [preds]
        return [float(p["e"]) * len(s) for p, s in zip(preds, structs)]

    def relax(self, s, fmax, steps):
        from chgnet.model import StructOptimizer
        r = StructOptimizer(model=self.model, use_device=self.device).relax(
            s, fmax=fmax, steps=steps, relax_cell=False, verbose=False)
        tr = r["trajectory"]
        f = float(np.linalg.norm(np.array(tr.forces[-1]), axis=1).max())
        return float(tr.energies[-1]), r["final_structure"], len(tr.energies), f

    def water(self, fmax=0.02):
        mol = Molecule(["O", "H", "H"], [[0, 0, 0], [0.757, 0.586, 0], [-0.757, 0.586, 0]])
        e, _, _, _ = self.relax(mol.get_boxed_structure(15, 15, 15), fmax, 300)
        return e


def gamma_J_m2(E, n_bulk_atoms, e_bulk_atom, n_water, e_water, area, dmu_water=0.0):
    """gamma = (E_slab - N e_bulk - n_H2O (E_H2O + dmu)) / 2A, in J/m^2.

    dmu_water = mu_H2O - E_H2O (eV per molecule). dmu = 0 takes the water from an isolated molecule at 0 K
    (no thermal energy, no entropy), which is the most favourable water there can be and makes strongly
    hydrated faces come out too low, even negative. Stored columns gamma_*_J_m2 keep dmu = 0 so they stay
    comparable with earlier runs; the shape (step 8) uses the condition in config["water_dmu_eV"]."""
    return (E - n_bulk_atoms * e_bulk_atom - n_water * (e_water + dmu_water)) / (2 * area) * EV_A2_TO_J_M2


# Water chemical potential relative to the 0 K molecule, eV per molecule (no zero-point energy on either
# side, as usual in ab initio thermodynamics). NIST-JANAF H-064: H(298.15)-H(0) = 9.904 kJ/mol,
# S(298.15) = 188.834 J/mol/K, S(400) = 198.788, H(400)-H(298.15) = 3.452; CODATA: Gibbs energy of
# vaporisation at 298.15 K = 8.547 kJ/mol. dmu(T, p) = [H(T) - H(0)] - T S(T) + kT ln(p / 1 bar).
WATER_DMU_EV = {
    "0K_molecule": 0.0,
    "vapour_298K_1bar": -0.4809,
    "liquid_298K": -0.5695,
    "vapour_298K_RH50": -0.5874,
    "vapour_400K_1bar": -0.6857,
}


def water_slope(n_water, area):
    """d gamma / d dmu = -n_H2O / 2A, returned as a positive number in J/m^2 per eV."""
    return n_water / (2 * area) * EV_A2_TO_J_M2


def gamma_at_mu(r, key, dmu):
    """gamma of a slab record at water chemical potential dmu, from its stored dmu = 0 value."""
    g = r.get(key)
    if g is None or (isinstance(g, float) and math.isnan(g)):
        return None
    return g - water_slope(r.get("n_water") or 0, r["area_A2"]) * dmu


# ----------------------------------------------------------------------------- step 8
def wulff(conv, families, gammas, rotations):
    """Wulff polyhedron from the true point group (not the lattice symmetry).

    families: list of family dicts (members include every symmetry-equivalent hkl and -hkl)
    gammas:   {family key: gamma}. Each member plane: n.x <= gamma, n = unit reciprocal vector.
    Returns dict with vertices, faces [(polygon, family key)], area fractions, volume,
    principal extents, bounded flag (False -> open along a direction: needle/rod limit).
    """
    from scipy.optimize import linprog
    from scipy.spatial import ConvexHull, HalfspaceIntersection
    R = conv.lattice.reciprocal_lattice_crystallographic
    planes = []
    for f in families:
        if f["key"] not in gammas or not np.isfinite(gammas[f["key"]]):
            continue
        g = gammas[f["key"]]
        for h in f["members"]:
            n = R.get_cartesian_coords(h); n = n / np.linalg.norm(n)
            planes.append((n, g, f["key"]))
    if not planes:
        return None
    gmax = max(p[1] for p in planes)
    A = np.array([p[0] for p in planes]); bvec = np.array([p[1] for p in planes])
    open_dirs = []
    for d in DIRS[::4]:
        res = linprog(-d, A_ub=A, b_ub=bvec, bounds=[(None, None)] * 3, method="highs")
        if res.status == 3:
            open_dirs.append(d)
    bounded = not open_dirs
    if not bounded:   # clip with a box so it can be drawn; clipped faces are labelled 'open'
        L = 6 * gmax
        for n in np.eye(3):
            for s in (1, -1):
                planes.append((s * n, L, "open"))
    hs = np.array([[*n, -g] for n, g, _ in planes])
    V = HalfspaceIntersection(hs, np.zeros(3)).intersections
    V = np.unique(np.round(V, 8), axis=0)
    hull = ConvexHull(V)
    faces, area = [], Counter()
    for n, g, key in planes:
        on = V[np.abs(V @ n - g) < 1e-5 * max(1, g)]
        if len(on) < 3:
            continue
        ctr = on.mean(0); u = on[0] - ctr; u /= np.linalg.norm(u); w = np.cross(n, u)
        poly = on[np.argsort(np.arctan2((on - ctr) @ w, (on - ctr) @ u))]
        a = 0.5 * np.linalg.norm(sum(np.cross(poly[i], poly[(i + 1) % len(poly)]) for i in range(len(poly))))
        if a < 1e-9:
            continue
        faces.append((poly, key)); area[key] += a
    tot = sum(v for k, v in area.items() if k != "open")
    frac = {k: v / sum(area.values()) for k, v in area.items()}
    ext = np.linalg.svd(V - V.mean(0), compute_uv=False)
    axes = np.linalg.svd(V - V.mean(0))[2]
    return dict(vertices=V, faces=faces, area_fractions=frac, volume=hull.volume, extents=ext, axes=axes,
                bounded=bounded, open_dirs=[d.tolist() for d in open_dirs])


def name_habit(w, families, crystal_system, conv):
    """Shape name from area fractions (cubic: polyhedron names; others: habit + dominant faces)."""
    if w is None:
        return "no faces"
    fam_name = {f["key"]: f["name"] for f in families}
    fr = {fam_name.get(k, k): v for k, v in w["area_fractions"].items()}
    if not w["bounded"]:
        d = np.array(w["open_dirs"][0]); fd = conv.lattice.get_fractional_coords(d)
        # K6 (15 Nov 2026): an unbounded Wulff shape means a face family that would close the crystal is
        # missing from the list (too few families kept, or no energy for it); it is not a rod.
        return f"open: no face closes the crystal along [{', '.join(f'{x:.2f}' for x in fd / np.abs(fd).max())}] (a face family is missing)"
    if crystal_system == "cubic":
        f = lambda k: fr.get(k, 0.0)
        # K1 (15 Nov 2026): the pure shapes and the cuboctahedron are single points of the map, so the names
        # are given only within numerical noise (PURE), not within 5 %; everything else is named by its family.
        PURE = 1e-3
        if f("{100}") > 1 - PURE: return "cube"
        if f("{111}") > 1 - PURE: return "octahedron"
        if f("{110}") > 1 - PURE: return "rhombic_dodecahedron"
        if f("{110}") < PURE and f("{100}") + f("{111}") > 1 - PURE:
            # {100} + {111} only: one-parameter series cube -> octahedron (project note, Step 11).
            # The cuboctahedron is the single point with 6 squares + 8 triangles of equal edge:
            # {100} share = 6 / (6 + 2*sqrt(3)) = 0.634. Below it: truncated octahedron; above: truncated cube.
            c = f("{100}") / (f("{100}") + f("{111}"))
            if abs(c - 6 / (6 + 2 * 3 ** 0.5)) < 0.005: return "cuboctahedron"
            return "truncated_octahedron" if c < 0.634 else "truncated_cube"
        top = max(fr, key=fr.get)
        if top == "{110}": return "truncated_rhombic_dodecahedron"
        return f"mixed_dominant_{top}"
    # Zingg (1935) classes, as in the project note (Part B, Step 10): b/a < 2/3 -> elongated, c/b < 2/3 -> flat
    e1, e2, e3 = w["extents"]
    long_, flat = e2 / e1 < 2 / 3, e3 / e2 < 2 / 3
    habit = "lath" if long_ and flat else "needle/rod" if long_ else "plate" if flat else "block/prism"
    top = sorted(fr.items(), key=lambda kv: -kv[1])[:3]
    return f"{habit}: " + ", ".join(f"{k} {v:.0%}" for k, v in top)


COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#b04fd1", "#d4a017", "#5b6770", "#e0457b", "#2bb3c0"]


def render(w, families, title, path):
    import matplotlib; import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    keys = [f["key"] for f in families]; names = {f["key"]: f["name"] for f in families}
    col = {k: COLORS[i % len(COLORS)] for i, k in enumerate(keys)}; col["open"] = "#d9d9d9"
    fig = plt.figure(figsize=(5, 5)); ax = fig.add_subplot(111, projection="3d")
    s = np.abs(w["vertices"]).max()
    for poly, k in w["faces"]:
        ax.add_collection3d(Poly3DCollection([poly / s], facecolor=col[k], edgecolor="#2b2b29", lw=0.6, alpha=0.95))
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1); ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=22, azim=35); ax.set_axis_off(); ax.set_title(title, fontsize=9)
    used = sorted({k for _, k in w["faces"]}, key=lambda k: -w["area_fractions"].get(k, 0))
    ax.legend([plt.Rectangle((0, 0), 1, 1, color=col[k]) for k in used],
              [f"{names.get(k, 'open (not computed)')} {w['area_fractions'][k]:.0%}" for k in used],
              loc="lower left", fontsize=7, frameon=False)
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)
    return path


def write_obj(w, path):
    lines, k = [], 1
    for poly, key in w["faces"]:
        lines.append(f"g {key}")
        for v in poly:
            lines.append("v " + " ".join(f"{x:.6f}" for x in v))
        lines.append("f " + " ".join(str(k + i) for i in range(len(poly)))); k += len(poly)
    Path(path).write_text("\n".join(lines))


# ----------------------------------------------------------------------------- step 7 helpers
# unpaired d electrons of the usual MOF oxidation states (high spin); K9 (15 Nov 2026)
UNPAIRED = {"Cu": 1, "Co": 3, "Ni": 2, "Fe": 5, "Mn": 5, "Cr": 3, "V": 1, "Ti": 0, "Zn": 0, "Zr": 0, "Hf": 0,
            "Mg": 0, "Ca": 0, "Al": 0, "Cd": 0, "In": 0, "Sc": 0}


def cp2k_input(struct, project, charge=0, slab=False):
    """Minimal CP2K PBE-D3(BJ) single-point input (DZVP-MOLOPT-SR, GTH) for a slab or bulk cell.

    K9 (15 Nov 2026): a slab is turned so that its normal (the third lattice vector, perpendicular to the
    first two) lies along z, and the surface dipole correction is switched on, because a slab with two
    different terminations carries a dipole that the periodic cell would otherwise repeat. Open-shell
    metals get a ferromagnetic high-spin multiplicity from UNPAIRED (Cu2+ d9: one unpaired electron each).
    The true ground state of the Cu2 paddlewheel is antiferromagnetic; using the same ferromagnetic state
    for bulk and slab makes most of that difference cancel in gamma. A metal not in UNPAIRED is flagged."""
    if slab:
        L = struct.lattice
        struct = Structure(Lattice.from_parameters(*L.abc, *L.angles), struct.species, struct.frac_coords)
    els = sorted({x.specie.symbol for x in struct})
    unknown = sorted({e for e in els if is_metal(e) and e not in UNPAIRED})
    n_up = sum(UNPAIRED.get(x.specie.symbol, 0) for x in struct)
    spin = f"    MULTIPLICITY {n_up + 1}\n" if n_up else ""
    if unknown:
        spin += f"    ! unpaired electrons of {', '.join(unknown)} not known: set MULTIPLICITY by hand\n"
    dip = "    SURFACE_DIPOLE_CORRECTION .TRUE.\n    SURF_DIP_DIR Z\n" if slab else ""
    kinds = "\n".join(f"    &KIND {e}\n      BASIS_SET DZVP-MOLOPT-SR-GTH\n      POTENTIAL GTH-PBE\n    &END KIND" for e in els)
    a, b, c = struct.lattice.matrix
    xyz = "\n".join(f"      {x.specie.symbol} {x.coords[0]:.6f} {x.coords[1]:.6f} {x.coords[2]:.6f}" for x in struct)
    return f"""&GLOBAL
  PROJECT {project}
  RUN_TYPE ENERGY
&END GLOBAL
&FORCE_EVAL
  METHOD Quickstep
  &DFT
    BASIS_SET_FILE_NAME BASIS_MOLOPT
    POTENTIAL_FILE_NAME GTH_POTENTIALS
    CHARGE {charge}
    UKS .TRUE.
{spin}{dip}    &MGRID
      CUTOFF 500
      REL_CUTOFF 50
    &END MGRID
    &SCF
      EPS_SCF 1.0E-6
      MAX_SCF 50
      &OT
        MINIMIZER DIIS
      &END OT
      &OUTER_SCF
        MAX_SCF 20
      &END OUTER_SCF
    &END SCF
    &XC
      &XC_FUNCTIONAL PBE
      &END XC_FUNCTIONAL
      &VDW_POTENTIAL
        POTENTIAL_TYPE PAIR_POTENTIAL
        &PAIR_POTENTIAL
          TYPE DFTD3(BJ)
          PARAMETER_FILE_NAME dftd3.dat
          REFERENCE_FUNCTIONAL PBE
        &END PAIR_POTENTIAL
      &END VDW_POTENTIAL
    &END XC
  &END DFT
  &SUBSYS
    &CELL
      A {a[0]:.6f} {a[1]:.6f} {a[2]:.6f}
      B {b[0]:.6f} {b[1]:.6f} {b[2]:.6f}
      C {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}
    &END CELL
    &COORD
{xyz}
    &END COORD
{kinds}
  &END SUBSYS
&END FORCE_EVAL
"""


# ----------------------------------------------------------------------------- driver
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


DEFAULTS = dict(symprec=0.1, disorder="majority", max_index=1, n_families=6, faces=None,
                thickness=15.0, vacuum=15.0, level_tol=0.3, max_terms=8, cap="water",
                relax_steps=800, fmax=0.05, relax_top_k=2,
                water_dmu_eV=WATER_DMU_EV["liquid_298K"], one_proton_per_donor=True)   # water reference used for the shape (step 8)


def _hkey(st):
    return hashlib.md5(st.to(fmt="poscar").encode()).hexdigest()[:10]


def run_mof(engine, name, cif, out, **cfg):
    """Steps 1-6 and 8 for one MOF. Everything is cached under out/cache/<name>/."""
    c = {**DEFAULTS, **{k: v for k, v in cfg.items() if v is not None}}
    out = Path(out); cdir = out / "cache" / re.sub(r"[^\w.-]", "_", name); cdir.mkdir(parents=True, exist_ok=True)
    sdir = out / "structures" / re.sub(r"[^\w.-]", "_", name); sdir.mkdir(parents=True, exist_ok=True)
    get = lambda k: json.loads((cdir / f"{k}.json").read_text()) if (cdir / f"{k}.json").exists() else None
    put = lambda k, v: (cdir / f"{k}.json").write_text(json.dumps(v, indent=1, default=str))

    # Step 1
    info = standardize(cif, c["symprec"], c["disorder"])
    log(f"{name}: {info['formula']}  SG {info['space_group']} ({info['symmetry_source']})  "
        f"conv {info['n_atoms_conv']} / working {info['n_atoms_prim']} atoms")
    for n in info["notes"]:
        log("   step1:", n)
    # Step 2
    blocks, _ = building_blocks(info["prim"])
    bs = summarize_blocks(blocks)
    log("   step2 blocks:", bs["blocks"], "| infinite inorganic unit" if bs["infinite_inorganic_unit"] else "")
    # Step 3 (families)
    fams = face_families(info["conv"], info["rotations"], c["max_index"], c["n_families"], c["faces"], ops=info["ops"])
    log("   step3 families:", ", ".join(f"{f['name']}(d={f['d_hkl']:.1f})" for f in fams))
    meta = dict(mof=name, cif=str(cif), formula=info["formula"], space_group=info["space_group"],
                crystal_system=info["crystal_system"], symmetry_source=info["symmetry_source"],
                conv_lattice=info["conv"].lattice.matrix.tolist(), notes=info["notes"], **bs,
                families=[{k: f[k] for k in ("key", "name", "rep", "d_hkl")} | {"n_members": len(f["members"])} for f in fams],
                config=c, _rotations=[R.tolist() for R in info["rotations"]])
    put("meta", meta)

    # Step 6a: bulk reference (same model, same settings)
    bkey = f"bulk_r{c['relax_steps']}_f{c['fmax']}"
    bulk = get(bkey)
    prim = info["prim"]
    if bulk is None:
        t = time.time()
        if c["relax_steps"] > 0:
            e, prim_r, n, fm = engine.relax(prim, c["fmax"], c["relax_steps"])
        else:
            e, prim_r, n, fm = engine.static([prim])[0], prim, 0, None
        prim_r.to(filename=str(sdir / f"{bkey}.cif"))
        bulk = dict(n_atoms=len(prim), energy_eV=e, e_per_atom_eV=e / len(prim), steps=n, final_fmax=fm,
                    converged=fm is None or fm <= c["fmax"] + 1e-9, seconds=round(time.time() - t, 1))
        put(bkey, bulk)
    prim_r = Structure.from_file(str(sdir / f"{bkey}.cif")) if c["relax_steps"] > 0 else prim
    # K11 fix (10 Nov 2026): the CIF round trip re-orients the lattice; put the relaxed fractional
    # coordinates back on the original lattice matrix so conv and prim share one frame.
    if c["relax_steps"] > 0:
        prim_r = Structure(prim.lattice, [x.species for x in prim_r], prim_r.frac_coords)
    bulk["cif"] = str(sdir / f"{bkey}.cif")
    conv = prim_r if info["conv"] is info["prim"] else info["conv"]
    e_bulk = bulk["e_per_atom_eV"]
    log(f"   step6 bulk: {e_bulk:.4f} eV/atom, converged {bulk['converged']}")
    modes = ["bare", "water"] if c["cap"] == "both" else (["water"] if c["cap"] == "water" else ["bare"])
    e_w = 0.0
    if "water" in modes:
        w = get("water") or {"E_H2O_eV": engine.water()}
        put("water", w); e_w = w["E_H2O_eV"]

    recs = []
    for f in fams:
        # Steps 3-4: terminations and slabs
        try:
            made = block_slabs(conv, prim_r, f["rep"], c["thickness"], c["vacuum"], c["level_tol"])
        except Exception as exc:
            log(f"   {f['name']}: slab construction failed: {exc}"); continue
        for m in made:
            miss, tot = broken_bonds(m)
            m["miss"], m["broken"] = miss, tot
            m["bond_density"] = sum(v for k, v in tot.items() if k in ("ML", "MO")) / 2 / m["area_A2"]
        n_all = len(made)
        made.sort(key=lambda m: m["bond_density"])
        made = made[: c["max_terms"]]            # rule: keep the terminations that break fewest bonds per area
        log(f"   {f['name']}: {n_all} distinct terminations, kept {len(made)}"
            + (" (cuts infinite inorganic unit)" if made and made[0]["cuts_infinite_unit"] else ""))
        for mode in modes:
            todo = []
            for ti, m in enumerate(made):
                if mode == "water":
                    st, n_w, status = heal(m, m["miss"], one_proton_per_donor=c["one_proton_per_donor"])
                else:
                    bad = m["broken"].get("org", 0) + m["broken"].get("MM", 0) + m["broken"].get("_extra", 0)
                    st, n_w, status = (None, 0, "rejected: non metal-donor bond broken") if bad else (m["slab"], 0, "bare")
                base = dict(mof=name, family=f["key"], face=f["name"], hkl=str(f["rep"]), termination=ti,
                            z0_A=m["z0"], cap=mode, status=status, n_slab_atoms=len(m["slab"]),
                            n_atoms=len(st) if st else None, n_water=n_w, area_A2=round(m["area_A2"], 2),
                            thickness_A=m["thickness_A"], n_layers=m["n_layers"],
                            broken_bonds=json.dumps(m["broken"]), bonds_per_A2=round(m["bond_density"], 5),
                            cuts_infinite_unit=m["cuts_infinite_unit"])
                if st is None:
                    recs.append(base); continue
                key = f"slab_{f['key']}_{mode}_{_hkey(st)}"
                todo.append((key, st, base, m))
            # static screen (batched)
            need = [t for t in todo if get(t[0] + "_static") is None]
            if need:
                es = engine.static([t[1] for t in need])
                for (key, st, base, m), e in zip(need, es):
                    put(key + "_static", {"E": e})
            for key, st, base, m in todo:
                e = get(key + "_static")["E"]
                base["gamma_static_J_m2"] = round(gamma_J_m2(e, len(m["slab"]), e_bulk, base["n_water"], e_w, m["area_A2"]), 4)
                base["_key"] = key; base["_st"] = st; base["_m"] = m
            # Step 6b: relax the K lowest static terminations (rule documented in DEFAULTS / README)
            # ranked at the water reference of step 8, so the terminations relaxed are the ones the shape uses
            dmu_rank = c["water_dmu_eV"] if mode == "water" else 0.0
            for t in todo:
                t[2]["water_slope_J_m2_per_eV"] = round(water_slope(t[2]["n_water"], t[3]["area_A2"]), 5)
            todo.sort(key=lambda t: gamma_at_mu(t[2], "gamma_static_J_m2", dmu_rank))
            for rank, (key, st, base, m) in enumerate(todo):
                if c["relax_steps"] > 0 and rank < c["relax_top_k"]:
                    rk = f"{key}_r{c['relax_steps']}_f{c['fmax']}"
                    r = get(rk)
                    if r is None:
                        t = time.time()
                        try:
                            E, fin, n, fm = engine.relax(st, c["fmax"], c["relax_steps"])
                        except Exception as exc:
                            log(f"      relax failed {key}: {exc}"); continue
                        fin.to(filename=str(sdir / f"{rk}.cif"))
                        r = dict(E=E, steps=n, final_fmax=fm, converged=fm <= c["fmax"] + 1e-9,
                                 seconds=round(time.time() - t, 1))
                        put(rk, r)
                    base.update(gamma_relaxed_J_m2=round(gamma_J_m2(r["E"], len(m["slab"]), e_bulk, base["n_water"], e_w, m["area_A2"]), 4),
                                steps=r["steps"], final_fmax=round(r["final_fmax"], 4), converged=r["converged"],
                                relax_seconds=r["seconds"], relaxed_cif=str(sdir / f"{rk}.cif"))
                    if not r["converged"]:
                        log(f"      NOT converged: {f['name']} t{base['termination']} {mode} fmax {r['final_fmax']:.3f}")
                st.to(filename=str(sdir / f"{key}_unrelaxed.cif"))
                base["unrelaxed_cif"] = str(sdir / f"{key}_unrelaxed.cif")
                log(f"      {f['name']} t{base['termination']} {mode:5s}: {base['n_atoms']} atoms, "
                    f"gamma static {base['gamma_static_J_m2']:.3f}"
                    + (f", relaxed {base['gamma_relaxed_J_m2']:.3f} ({base['steps']} steps)" if 'gamma_relaxed_J_m2' in base else ""))
                recs.append({k: v for k, v in base.items() if not k.startswith("_")})
    return meta, fams, recs, bulk


def choose_gamma(recs, mode, need_relaxed=True, dmu=0.0):
    """Lowest gamma per family at water chemical potential dmu: relaxed + converged if available
    (required when need_relaxed), otherwise static (flagged). The lowest termination can change with dmu,
    because terminations differ in water per area."""
    best = {}
    for r in recs:
        if r.get("cap") != mode or r.get("gamma_static_J_m2") is None:
            continue
        d = dmu if mode == "water" else 0.0
        if r.get("gamma_relaxed_J_m2") is not None and r.get("converged"):
            g, src = gamma_at_mu(r, "gamma_relaxed_J_m2", d), "relaxed"
        elif need_relaxed:
            continue
        else:
            g, src = gamma_at_mu(r, "gamma_static_J_m2", d), "static"
        k = r["family"]
        if k not in best or g < best[k]["gamma"]:
            best[k] = dict(gamma=g, source=src, termination=r["termination"], face=r["face"])
    return best


def wulff_report(meta, fams, recs, out, mode, need_relaxed=True, dmu=None):
    conv = Structure(Lattice(meta["conv_lattice"]), ["H"], [[0, 0, 0]])
    dmu = meta.get("config", {}).get("water_dmu_eV", 0.0) if dmu is None else dmu
    best = choose_gamma(recs, mode, need_relaxed, dmu)
    best0 = choose_gamma(recs, mode, need_relaxed, 0.0)
    row = dict(mof=meta["mof"], cap=mode, space_group=meta["space_group"], n_families_with_gamma=len(best),
               water_dmu_eV=dmu if mode == "water" else None,
               **{f"gamma0K_{k}": round(b["gamma"], 4) for k, b in best0.items()},
               gamma_source=",".join(sorted({b["source"] for b in best.values()})) or "none",
               **{f"gamma_{k}": round(b["gamma"], 4) for k, b in best.items()})
    if len(best) < 2:
        row["habit"] = "not enough faces"; return row, None
    if any(b["gamma"] <= 0 for b in best.values()):
        row["habit"] = "undefined: gamma <= 0 for " + ",".join(b["face"] for b in best.values() if b["gamma"] <= 0) \
                       + f" at water dmu = {dmu:+.3f} eV (hydrating the cut is downhill at this water reference)"
        return row, None
    rots = [np.array(r) for r in meta["_rotations"]]
    w = wulff(conv, fams, {k: b["gamma"] for k, b in best.items()}, rots)
    name = name_habit(w, fams, meta["crystal_system"], conv)
    fr = {next(f["name"] for f in fams if f["key"] == k) if k != "open" else "open": round(v, 3)
          for k, v in sorted(w["area_fractions"].items(), key=lambda kv: -kv[1])}
    e = w["extents"]
    row.update(habit=name, area_fractions=json.dumps(fr), dominant_face=max(fr, key=fr.get),
               bounded=w["bounded"], aspect_e1_e2=round(e[0] / e[1], 3), aspect_e2_e3=round(e[1] / e[2], 3))
    tag = re.sub(r"[^\w.-]", "_", meta["mof"])
    png = render(w, fams, f"{meta['mof']} ({mode})\n{name}", Path(out) / f"wulff_{tag}_{mode}.png")
    write_obj(w, Path(out) / f"wulff_{tag}_{mode}.obj")
    row["png"] = str(png)
    return row, w


## Input MOFs

Edit the table. `source` can be:
- a URL to a CIF (defaults: RASPA2 structure library on GitHub),
- `core:REFCODE`, which takes the structure from CoRE MOF 2019 (Zenodo; downloads the ~81 MB archive once), e.g. `core:RUBTAK` (UiO-66), `core:FIQCEN` (HKUST-1), `core:VOGTIV` (Mg-MOF-74), `core:OFERUN` (ZIF-8),
- `upload`, which asks for the file.

`faces` (optional) overrides the automatic family list, e.g. `"100 110 111"`. `reference_habit` is only displayed next to the prediction. Defaults are commonly reported additive-free habits; check them against your own sources. Real comparison against curated data is step 9.

Note: the RASPA2 UiO-66 model has dehydroxylated Zr6O6 nodes. Use `core:RUBTAK` for the hydroxylated cluster.

In [ ]:
# 3. MOF list + download
import pandas as pd, requests, zipfile, glob
RASPA = "https://raw.githubusercontent.com/iRASPA/RASPA2/master/structures/mofs/cif/"
MOFS = pd.DataFrame([
    dict(mof="ZIF-8",     source=RASPA + "ZIF-8.cif",    faces="", reference_habit="rhombic dodecahedron {110}"),
    dict(mof="HKUST-1",   source=RASPA + "Cu-BTC.cif",   faces="", reference_habit="octahedron {111}"),
    dict(mof="Mg-MOF-74", source=RASPA + "MgMOF-74.cif", faces="", reference_habit="hexagonal rods/needles along c"),
    dict(mof="UiO-66",    source="embedded:UiO-66",   faces="", reference_habit="cube {100} without additives; octahedron {111} only with benzoic or acetic acid"),
])
CORE_URL = "https://zenodo.org/records/14184621/files/CoREMOF2019_public_v2_20241119.zip?download=1"
CIF_DIR = os.path.join(BASE, "cifs"); os.makedirs(CIF_DIR, exist_ok=True)

def fetch(url, path):
    if not os.path.exists(path):
        r = requests.get(url, timeout=300, stream=True); r.raise_for_status()
        with open(path + ".part", "wb") as f:
            for b in r.iter_content(1 << 20): f.write(b)
        os.replace(path + ".part", path)
    return path

EMBEDDED = {"UiO-66": "#======================================================================\n\n# CRYSTAL DATA\n\n#----------------------------------------------------------------------\n\ndata_VESTA_phase_1\n\n\n_pd_phase_name                         'UiO-66 - PBEsol                       '\n_cell_length_a                         14.50863\n_cell_length_b                         14.50863\n_cell_length_c                         14.50863\n_cell_angle_alpha                      60.00000\n_cell_angle_beta                       60.00000\n_cell_angle_gamma                      60.00000\n_symmetry_space_group_name_H-M         'P 1'\n_symmetry_Int_Tables_number            1\n\nloop_\n_symmetry_equiv_pos_as_xyz\n   'x, y, z'\n\nloop_\n   _atom_site_label\n   _atom_site_occupancy\n   _atom_site_fract_x\n   _atom_site_fract_y\n   _atom_site_fract_z\n   _atom_site_adp_type\n   _atom_site_B_iso_or_equiv\n   _atom_site_type_symbol\n   H1         1.0     0.155217      0.580411      0.404711     Biso  1.000000 H\n   H2         1.0     0.580411      0.155217      0.859660     Biso  1.000000 H\n   H3         1.0     0.404711      0.859660      0.155217     Biso  1.000000 H\n   H4         1.0     0.859660      0.404711      0.580411     Biso  1.000000 H\n   H5         1.0     0.404711      0.155217      0.580411     Biso  1.000000 H\n   H6         1.0     0.859660      0.580411      0.155217     Biso  1.000000 H\n   H7         1.0     0.155217      0.404711      0.859660     Biso  1.000000 H\n   H8         1.0     0.580411      0.859660      0.404711     Biso  1.000000 H\n   H9         1.0     0.580411      0.404711      0.155217     Biso  1.000000 H\n   H10        1.0     0.155217      0.859660      0.580411     Biso  1.000000 H\n   H11        1.0     0.859660      0.155217      0.404711     Biso  1.000000 H\n   H12        1.0     0.404711      0.580411      0.859660     Biso  1.000000 H\n   H13        1.0     0.580411      0.155217      0.404711     Biso  1.000000 H\n   H14        1.0     0.155217      0.580411      0.859660     Biso  1.000000 H\n   H15        1.0     0.859660      0.404711      0.155217     Biso  1.000000 H\n   H16        1.0     0.404711      0.859660      0.580411     Biso  1.000000 H\n   H17        1.0     0.155217      0.404711      0.580411     Biso  1.000000 H\n   H18        1.0     0.580411      0.859660      0.155217     Biso  1.000000 H\n   H19        1.0     0.404711      0.155217      0.859660     Biso  1.000000 H\n   H20        1.0     0.859660      0.580411      0.404711     Biso  1.000000 H\n   H21        1.0     0.404711      0.580411      0.155217     Biso  1.000000 H\n   H22        1.0     0.859660      0.155217      0.580411     Biso  1.000000 H\n   H23        1.0     0.155217      0.859660      0.404711     Biso  1.000000 H\n   H24        1.0     0.580411      0.404711      0.859660     Biso  1.000000 H\n   H25        1.0     0.904736      0.904736      0.285792     Biso  1.000000 H\n   H26        1.0     0.904736      0.904736      0.904736     Biso  1.000000 H\n   H27        1.0     0.285792      0.904736      0.904736     Biso  1.000000 H\n   H28        1.0     0.904736      0.285792      0.904736     Biso  1.000000 H\n   C1         1.0     0.091988      0.539726      0.444553     Biso  1.000000 C\n   C2         1.0     0.539726      0.091988      0.923733     Biso  1.000000 C\n   C3         1.0     0.444553      0.923733      0.091988     Biso  1.000000 C\n   C4         1.0     0.923733      0.444553      0.539726     Biso  1.000000 C\n   C5         1.0     0.444553      0.091988      0.539726     Biso  1.000000 C\n   C6         1.0     0.923733      0.539726      0.091988     Biso  1.000000 C\n   C7         1.0     0.091988      0.444553      0.923733     Biso  1.000000 C\n   C8         1.0     0.539726      0.923733      0.444553     Biso  1.000000 C\n   C9         1.0     0.539726      0.444553      0.091988     Biso  1.000000 C\n   C10        1.0     0.091988      0.923733      0.539726     Biso  1.000000 C\n   C11        1.0     0.923733      0.091988      0.444553     Biso  1.000000 C\n   C12        1.0     0.444553      0.539726      0.923733     Biso  1.000000 C\n   C13        1.0     0.539726      0.091988      0.444553     Biso  1.000000 C\n   C14        1.0     0.091988      0.539726      0.923733     Biso  1.000000 C\n   C15        1.0     0.923733      0.444553      0.091988     Biso  1.000000 C\n   C16        1.0     0.444553      0.923733      0.539726     Biso  1.000000 C\n   C17        1.0     0.091988      0.444553      0.539726     Biso  1.000000 C\n   C18        1.0     0.539726      0.923733      0.091988     Biso  1.000000 C\n   C19        1.0     0.444553      0.091988      0.923733     Biso  1.000000 C\n   C20        1.0     0.923733      0.539726      0.444553     Biso  1.000000 C\n   C21        1.0     0.444553      0.539726      0.091988     Biso  1.000000 C\n   C22        1.0     0.923733      0.091988      0.539726     Biso  1.000000 C\n   C23        1.0     0.091988      0.923733      0.444553     Biso  1.000000 C\n   C24        1.0     0.539726      0.444553      0.923733     Biso  1.000000 C\n   C25        1.0     0.299965      0.004529      0.004529     Biso  1.000000 C\n   C26        1.0     0.004529      0.299965      0.690977     Biso  1.000000 C\n   C27        1.0     0.004529      0.690977      0.299965     Biso  1.000000 C\n   C28        1.0     0.690977      0.004529      0.004529     Biso  1.000000 C\n   C29        1.0     0.004529      0.299965      0.004529     Biso  1.000000 C\n   C30        1.0     0.690977      0.004529      0.299965     Biso  1.000000 C\n   C31        1.0     0.299965      0.004529      0.690977     Biso  1.000000 C\n   C32        1.0     0.004529      0.690977      0.004529     Biso  1.000000 C\n   C33        1.0     0.004529      0.004529      0.299965     Biso  1.000000 C\n   C34        1.0     0.299965      0.690977      0.004529     Biso  1.000000 C\n   C35        1.0     0.690977      0.299965      0.004529     Biso  1.000000 C\n   C36        1.0     0.004529      0.004529      0.690977     Biso  1.000000 C\n   C37        1.0     0.007411      0.586820      0.398358     Biso  1.000000 C\n   C38        1.0     0.586820      0.007411      0.007411     Biso  1.000000 C\n   C39        1.0     0.398358      0.007411      0.007411     Biso  1.000000 C\n   C40        1.0     0.007411      0.398358      0.586820     Biso  1.000000 C\n   C41        1.0     0.398358      0.007411      0.586820     Biso  1.000000 C\n   C42        1.0     0.007411      0.586820      0.007411     Biso  1.000000 C\n   C43        1.0     0.007411      0.398358      0.007411     Biso  1.000000 C\n   C44        1.0     0.586820      0.007411      0.398358     Biso  1.000000 C\n   C45        1.0     0.586820      0.398358      0.007411     Biso  1.000000 C\n   C46        1.0     0.007411      0.007411      0.586820     Biso  1.000000 C\n   C47        1.0     0.007411      0.007411      0.398358     Biso  1.000000 C\n   C48        1.0     0.398358      0.586820      0.007411     Biso  1.000000 C\n   O1         1.0     0.081082      0.731998      0.262517     Biso  1.000000 O\n   O2         1.0     0.731998      0.081082      0.924403     Biso  1.000000 O\n   O3         1.0     0.262517      0.924403      0.081082     Biso  1.000000 O\n   O4         1.0     0.924403      0.262517      0.731998     Biso  1.000000 O\n   O5         1.0     0.262517      0.081082      0.731998     Biso  1.000000 O\n   O6         1.0     0.924403      0.731998      0.081082     Biso  1.000000 O\n   O7         1.0     0.081082      0.262517      0.924403     Biso  1.000000 O\n   O8         1.0     0.731998      0.924403      0.262517     Biso  1.000000 O\n   O9         1.0     0.731998      0.262517      0.081082     Biso  1.000000 O\n   O10        1.0     0.081082      0.924403      0.731998     Biso  1.000000 O\n   O11        1.0     0.924403      0.081082      0.262517     Biso  1.000000 O\n   O12        1.0     0.262517      0.731998      0.924403     Biso  1.000000 O\n   O13        1.0     0.731998      0.081082      0.262517     Biso  1.000000 O\n   O14        1.0     0.081082      0.731998      0.924403     Biso  1.000000 O\n   O15        1.0     0.924403      0.262517      0.081082     Biso  1.000000 O\n   O16        1.0     0.262517      0.924403      0.731998     Biso  1.000000 O\n   O17        1.0     0.081082      0.262517      0.731998     Biso  1.000000 O\n   O18        1.0     0.731998      0.924403      0.081082     Biso  1.000000 O\n   O19        1.0     0.262517      0.081082      0.924403     Biso  1.000000 O\n   O20        1.0     0.924403      0.731998      0.262517     Biso  1.000000 O\n   O21        1.0     0.262517      0.731998      0.081082     Biso  1.000000 O\n   O22        1.0     0.924403      0.081082      0.731998     Biso  1.000000 O\n   O23        1.0     0.081082      0.924403      0.262517     Biso  1.000000 O\n   O24        1.0     0.731998      0.262517      0.924403     Biso  1.000000 O\n   O25        1.0     0.203690      0.932103      0.932103     Biso  1.000000 O\n   O26        1.0     0.932103      0.203690      0.932103     Biso  1.000000 O\n   O27        1.0     0.932103      0.932103      0.203690     Biso  1.000000 O\n   O28        1.0     0.932103      0.932103      0.932103     Biso  1.000000 O\n   O29        1.0     0.846778      0.051074      0.051074     Biso  1.000000 O\n   O30        1.0     0.051074      0.846778      0.051074     Biso  1.000000 O\n   O31        1.0     0.051074      0.051074      0.846778     Biso  1.000000 O\n   O32        1.0     0.051074      0.051074      0.051074     Biso  1.000000 O\n   Zr1        1.0     0.120756      0.879244      0.120756     Biso  1.000000 Zr\n   Zr2        1.0     0.879244      0.120756      0.879244     Biso  1.000000 Zr\n   Zr3        1.0     0.120756      0.120756      0.879244     Biso  1.000000 Zr\n   Zr4        1.0     0.879244      0.879244      0.120756     Biso  1.000000 Zr\n   Zr5        1.0     0.879244      0.120756      0.120756     Biso  1.000000 Zr\n   Zr6        1.0     0.120756      0.879244      0.879244     Biso  1.000000 Zr\n"}   # Zr6O4(OH)4 node (website/UiO-66.cif)

def get_cif(row):
    src = str(row["source"]).strip(); out = f"{CIF_DIR}/{row['mof']}.cif"
    if os.path.exists(out): return out
    if src.startswith("embedded:"):
        open(out, "w").write(EMBEDDED[src[9:]])
    elif src.startswith("core:"):
        ref = src[5:].upper(); z = fetch(CORE_URL, os.path.join(BASE, "CoREMOF2019.zip"))
        with zipfile.ZipFile(z) as zz:
            names = [n for n in zz.namelist() if os.path.basename(n).upper().startswith(ref) and n.endswith(".cif")]
            if not names: raise FileNotFoundError(f"{ref} not in CoRE MOF 2019")
            names.sort(key=lambda n: ("/CR/" not in n, "ASR" not in n, len(n)))
            open(out, "wb").write(zz.read(names[0])); print(f"{row['mof']}: {names[0]}")
    elif src == "upload":
        from google.colab import files
        print(f"upload the CIF for {row['mof']}"); up = files.upload(); open(out, "wb").write(list(up.values())[0])
    else:
        fetch(src, out)
    return out

CIF = {}
for _, r in MOFS.iterrows():
    try: CIF[r["mof"]] = get_cif(r); print("ok  ", r["mof"], CIF[r["mof"]])
    except Exception as e: print("FAIL", r["mof"], e)

In [ ]:
# 4. Settings
QUICK_TEST   = False   #@param {type:"boolean"}
ONLY         = ""      #@param {type:"string"}
CAP          = "water" #@param ["water", "bare", "both"]
RELAX_STEPS  = 800     #@param {type:"integer"}
FMAX         = 0.05    #@param {type:"number"}
RELAX_TOP_K  = 2       #@param {type:"integer"}
MAX_TERMS    = 8       #@param {type:"integer"}
N_FAMILIES   = 4       #@param {type:"integer"}
MAX_INDEX    = 1       #@param {type:"integer"}
THICKNESS    = 15.0    #@param {type:"number"}
VACUUM       = 15.0    #@param {type:"number"}
WATER_DMU_EV = -0.5695  #@param {type:"number"}
# Water condition for the shapes and the GNN labels: chemical potential of water relative to a free molecule
# at 0 K, in eV. -0.5695 = liquid water at 25 C (default); -0.4809 = vapour, 25 C, 1 bar; 0 = the old
# reference, which gave negative surface energies.
SYMPREC      = 0.1     #@param {type:"number"}
DISORDER     = "majority" #@param ["majority", "reject"]
# QUICK_TEST = static energies only (seconds per MOF). Not publishable; use it to check that a MOF runs.

In [ ]:
# 5. Steps 1-6 and 8
import sys, importlib, json, re
sys.path.insert(0, BASE)
import mofwulff as mw; importlib.reload(mw)
from IPython.display import Image, display

eng = mw.Engine()
only = {x.strip() for x in ONLY.split(",") if x.strip()}
cfg = dict(symprec=SYMPREC, disorder=DISORDER, max_index=MAX_INDEX, n_families=N_FAMILIES,
           thickness=THICKNESS, vacuum=VACUUM, max_terms=MAX_TERMS, cap=CAP,
           relax_steps=0 if QUICK_TEST else RELAX_STEPS, fmax=FMAX, relax_top_k=RELAX_TOP_K,
           water_dmu_eV=WATER_DMU_EV)
modes = ["bare", "water"] if CAP == "both" else [CAP if CAP == "water" else "bare"]
ALL, SUMMARY, RUNS = [], [], {}
for _, r in MOFS.iterrows():
    name = r["mof"]
    if only and name not in only or name not in CIF: continue
    faces = None
    if str(r.get("faces", "")).strip():
        faces = [tuple(int(d) for d in re.findall(r"-?\d", t)) for t in re.split(r"[;,\s]+", r["faces"].strip()) if t]
    try:
        meta, fams, recs, bulk = mw.run_mof(eng, name, CIF[name], OUT_DIR, faces=faces, **cfg)
    except Exception as e:
        print(f"{name}: FAILED {type(e).__name__}: {e}"); continue
    RUNS[name] = (meta, fams, recs, bulk); ALL += recs
    for mode in modes:
        row, w = mw.wulff_report(meta, fams, recs, OUT_DIR, mode, need_relaxed=not QUICK_TEST)
        row.update(reference_habit=r.get("reference_habit", ""), bulk_converged=bulk["converged"],
                   n_unconverged=sum(1 for x in recs if x.get("cap") == mode and x.get("converged") is False),
                   n_rejected=sum(1 for x in recs if x.get("cap") == mode and str(x.get("status", "")).startswith("rejected")),
                   notes="; ".join(meta["notes"]))
        if not bulk["converged"]: row["habit"] += "  [WARNING: bulk not converged]"
        SUMMARY.append(row)
        if row.get("png"): display(Image(row["png"]))
slabs = pd.DataFrame(ALL); slabs.to_csv(f"{OUT_DIR}/slabs.csv", index=False)
summary = pd.DataFrame(SUMMARY); summary.to_csv(f"{OUT_DIR}/wulff_summary.csv", index=False)
gcols = [c for c in summary.columns if c.startswith("gamma_")]
display(summary[["mof", "cap", "habit", "dominant_face", "area_fractions", "reference_habit", "gamma_source",
                 "bulk_converged", "n_unconverged", "n_rejected"] + gcols])
deliver("steps1-8")


In [ ]:
# 6. Surface-energy table (every termination). gamma in J/m^2.
cols = ["mof", "face", "termination", "cap", "status", "n_atoms", "n_water", "bonds_per_A2",
        "gamma_static_J_m2", "gamma_relaxed_J_m2", "steps", "final_fmax", "converged", "cuts_infinite_unit"]
display(slabs[[c for c in cols if c in slabs.columns]].sort_values(["mof", "cap", "face", "termination"]))

In [ ]:
# 5b. Slab thickness check: does a thicker slab change gamma? (one face; about 20 min on a T4)
THICKNESS_CHECK = True  #@param {type:"boolean"}
CHECK_MOF, CHECK_FACE = "ZIF-8", (1, 1, 0)
if THICKNESS_CHECK and CHECK_MOF in CIF and not QUICK_TEST:
    rows = []
    for t in (15.0, 22.0, 30.0):
        cfg_t = dict(cfg, faces=[CHECK_FACE], thickness=t, relax_top_k=1)
        meta_t, fams_t, recs_t, bulk_t = mw.run_mof(eng, CHECK_MOF, CIF[CHECK_MOF], f"{OUT_DIR}/thickness_{int(t)}A", **cfg_t)
        for r in recs_t:
            if r.get("gamma_relaxed_J_m2") is not None:
                rows.append(dict(min_thickness_A=t, face=r["face"], termination=r["termination"], n_atoms=r["n_atoms"],
                                 thickness_A=r["thickness_A"], gamma_static_J_m2=r["gamma_static_J_m2"],
                                 gamma_relaxed_0K_J_m2=r["gamma_relaxed_J_m2"],
                                 gamma_relaxed_at_mu_J_m2=mw.gamma_at_mu(r, "gamma_relaxed_J_m2", WATER_DMU_EV),
                                 converged=r["converged"]))
    THICK = pd.DataFrame(rows); THICK.to_csv(f"{OUT_DIR}/thickness_check.csv", index=False); display(THICK)
    print("Converged in thickness if gamma changes by less than about 0.01 J/m2 between 22 and 30 A.")


## Step 7: validation

**7a.** MACE-MP-0, a second universal MLIP trained independently of CHGNet, re-computes the energies of the CHGNet-relaxed bulk, water and slabs (single points). Reported per MOF: Spearman rank correlation of the face ordering, plus MAE in γ. Disagreement flags faces whose CHGNet energies should not be trusted. Agreement does **not** prove accuracy: both models share Materials Project training data.

**7b.** CP2K inputs (PBE-D3(BJ), DZVP-MOLOPT-SR, GTH, 500 Ry) for a small held-out set: the lowest-γ termination of the `VAL_FACES` smallest faces of each of the first `VAL_MOFS` MOFs, plus bulk and H₂O. Run them on an HPC cluster.

**7c.** Upload the DFT energies to get the validation table.

In [ ]:
# 7a. Independent-MLIP cross-check (MACE-MP-0)
RUN_MACE = True  #@param {type:"boolean"}
from scipy.stats import spearmanr
import numpy as np
from pymatgen.core import Structure
VAL7A = None
if RUN_MACE and len(slabs):
    if importlib.util.find_spec("mace") is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "mace-torch"], check=True)
    from mace.calculators import mace_mp
    from pymatgen.io.ase import AseAtomsAdaptor
    from ase.build import molecule
    from ase.optimize import BFGS
    calc = mace_mp(model="medium", default_dtype="float64", device="cuda" if torch.cuda.is_available() else "cpu")
    def E(st):
        a = AseAtomsAdaptor.get_atoms(st); a.calc = calc; return float(a.get_potential_energy())
    h2o = molecule("H2O"); h2o.center(vacuum=7.5); h2o.calc = calc; BFGS(h2o, logfile=None).run(fmax=0.02)
    e_w_mace = float(h2o.get_potential_energy())
    rows = []
    for name, (meta, fams, recs, bulk) in RUNS.items():
        b = Structure.from_file(bulk["cif"]); eb = E(b) / len(b)
        for x in recs:
            cif = x.get("relaxed_cif") if x.get("converged") else (x.get("unrelaxed_cif") if QUICK_TEST else None)
            if not cif: continue
            g_chg = x["gamma_relaxed_J_m2"] if x.get("converged") else x["gamma_static_J_m2"]
            st = Structure.from_file(cif)
            g_m = mw.gamma_J_m2(E(st), x["n_slab_atoms"], eb, x["n_water"], e_w_mace, x["area_A2"])
            rows.append(dict(mof=name, face=x["face"], termination=x["termination"], cap=x["cap"],
                             gamma_chgnet=g_chg, gamma_mace=round(g_m, 4)))
    VAL7A = pd.DataFrame(rows)
    if len(VAL7A):
        VAL7A.to_csv(f"{OUT_DIR}/validation_mace.csv", index=False)
        stats = []
        for (m, cap), g in VAL7A.groupby(["mof", "cap"]):
            best = g.loc[g.groupby("face")["gamma_chgnet"].idxmin()]
            rho = spearmanr(best["gamma_chgnet"], best["gamma_mace"]).correlation if len(best) > 2 else np.nan
            same_min = best.loc[best["gamma_chgnet"].idxmin(), "face"] == best.loc[best["gamma_mace"].idxmin(), "face"]
            rho_all = spearmanr(g["gamma_chgnet"], g["gamma_mace"]).correlation if len(g) > 2 else np.nan
            stats.append(dict(mof=m, cap=cap, n_faces=len(best), spearman_faces=rho, n_slabs=len(g), spearman_all_slabs=rho_all,
                              lowest_face_agrees=same_min,
                              MAE_J_m2=float(np.mean(np.abs(g["gamma_chgnet"] - g["gamma_mace"])))))
        display(VAL7A); display(pd.DataFrame(stats))

In [ ]:
# 7b. CP2K PBE-D3 inputs for a small held-out set
VAL_MOFS  = 2  #@param {type:"integer"}
VAL_FACES = 2  #@param {type:"integer"}
dft_dir = f"{OUT_DIR}/dft_inputs"; os.makedirs(dft_dir, exist_ok=True)
manifest = []
for name, (meta, fams, recs, bulk) in list(RUNS.items())[:VAL_MOFS]:
    tag = re.sub(r"[^\w.-]", "_", name)
    b = Structure.from_file(bulk["cif"])
    open(f"{dft_dir}/{tag}_bulk.inp", "w").write(mw.cp2k_input(b, f"{tag}_bulk"))
    manifest.append(dict(file=f"{tag}_bulk.inp", mof=name, kind="bulk", n_atoms=len(b)))
    ok = [x for x in recs if x.get("relaxed_cif") or (QUICK_TEST and x.get("unrelaxed_cif"))]
    best = {}
    for x in ok:
        g = x.get("gamma_relaxed_J_m2", x.get("gamma_static_J_m2"))
        if x["face"] not in best or g < best[x["face"]][0]: best[x["face"]] = (g, x)
    for face, (g, x) in sorted(best.items(), key=lambda kv: kv[1][1]["n_atoms"])[:VAL_FACES]:
        st = Structure.from_file(x.get("relaxed_cif") or x["unrelaxed_cif"])
        f = f"{tag}_{x['family']}_t{x['termination']}_{x['cap']}.inp"
        open(f"{dft_dir}/{f}", "w").write(mw.cp2k_input(st, f[:-4], slab=True))
        manifest.append(dict(file=f, mof=name, kind="slab", face=x["face"], termination=x["termination"], cap=x["cap"],
                             n_atoms=len(st), n_slab_atoms=x["n_slab_atoms"], n_water=x["n_water"], area_A2=x["area_A2"],
                             gamma_chgnet=g))
from pymatgen.core import Molecule
w = Molecule(["O", "H", "H"], [[0, 0, 0], [0.757, 0.586, 0], [-0.757, 0.586, 0]]).get_boxed_structure(15, 15, 15)
open(f"{dft_dir}/H2O.inp", "w").write(mw.cp2k_input(w, "H2O")); manifest.append(dict(file="H2O.inp", kind="water", n_atoms=3))
MANIFEST = pd.DataFrame(manifest); MANIFEST.to_csv(f"{dft_dir}/manifest.csv", index=False); display(MANIFEST)
print("Run each .inp with CP2K (cp2k.psmp -i X.inp). Optimise H2O.inp's geometry or use a single point.")

In [ ]:
# 7c. Upload DFT results: CSV with columns  file, energy  (energy in Hartree, as CP2K prints; or add column unit=eV)
UPLOAD_DFT = False  #@param {type:"boolean"}
if UPLOAD_DFT:
    from google.colab import files
    up = files.upload(); dft = pd.read_csv(list(up)[0])
    is_ev = dft["unit"].astype(str).str.lower().eq("ev") if "unit" in dft else False
    dft["E_eV"] = np.where(is_ev, dft["energy"], dft["energy"] * 27.211386)
    m = MANIFEST.merge(dft[["file", "E_eV"]], on="file", how="left")
    ew = m.loc[m["kind"] == "water", "E_eV"].iloc[0]
    out = []
    for name, g in m.groupby("mof"):
        b = g[g["kind"] == "bulk"].iloc[0]; eb = b["E_eV"] / b["n_atoms"]
        for _, s in g[g["kind"] == "slab"].iterrows():
            gd = mw.gamma_J_m2(s["E_eV"], s["n_slab_atoms"], eb, s["n_water"], ew, s["area_A2"]) if pd.notna(s["E_eV"]) else np.nan
            out.append(dict(mof=name, face=s["face"], cap=s["cap"], gamma_chgnet=s["gamma_chgnet"], gamma_dft=gd,
                            status="failed/missing" if np.isnan(gd) else "ok"))
    V = pd.DataFrame(out); V.to_csv(f"{OUT_DIR}/validation_dft.csv", index=False); display(V)
    okv = V.dropna()
    if len(okv) > 2:
        print("MAE (J/m2):", float(np.mean(np.abs(okv.gamma_chgnet - okv.gamma_dft))),
              "| Spearman:", spearmanr(okv.gamma_chgnet, okv.gamma_dft).correlation,
              "| failures:", int(V.gamma_dft.isna().sum()))

In [ ]:
# 8. Download everything so far (CSVs, Wulff PNG/OBJ, relaxed slab CIFs, DFT inputs) to your Downloads folder
deliver("after_validation")


## Limits that remain (steps 1–8)
- γ is the **average of the two faces** for slabs whose top and bottom terminations differ (`symmetric` is not enforced). For non-centrosymmetric MOFs (ZIF-8, I-43m) {111} and {1̅1̅1̅} are always averaged.
- The healing chemistry is one choice (dissociated water). Other caps (formate, DMF, modulator acid) change γ. That dependence on conditions is what steps 9–10 model.
- Static screening picks which terminations to relax. A termination ranked high statically but low after relaxation can be missed; raise `RELAX_TOP_K` to check.
- CHGNet is not validated for MOF surfaces. Treat habits as hypotheses until 7c is done.

# GNN surrogate for surface energies (option a)

**Role.** A fast student of the expensive teacher. The teacher is steps 4–6: build the slab, heal it, relax it with CHGNet, compute γ. The student is a graph neural network that reads the **unrelaxed healed slab** and predicts γ directly. Step 8 is unchanged: lowest γ per face family → Wulff → habit.

| | |
|---|---|
| Input | atom graph of the healed slab: nodes = atoms (element, metal / cap / donor flags, bond counts, depth below the surface), edges = the pipeline's own bond rules (org, M–M, M–ligand, M–oxo) with distance features |
| Output | γ per (face, termination), in J/m² |
| Readout | each atom gets an excess energy eᵢ; γ = Σeᵢ / 2A. Same form as γ = (E_slab − N·e_bulk − n·E_H₂O)/2A, so it does not depend on slab thickness |
| Labels | `gamma_relaxed_J_m2` from step 6 (converged slabs only) |
| Split | by MOF: leave-one-MOF-out up to 10 MOFs, grouped 5-fold above that |
| Baselines (every fold) | constant mean · linear fit on broken bonds per Å² · CHGNet static γ (raw and linearly calibrated) |

**Why the input is the unrelaxed slab.** Relaxation is the step the GNN is supposed to skip. A model trained on relaxed geometries would need a CHGNet relaxation before it could be used.

**`TARGET = "delta"`** learns γ_relaxed − γ_static instead. Each new slab then needs one CHGNet single point (seconds) but no relaxation. Try both. The GNN must beat the static-CHGNet baseline to be worth having, because that baseline already costs only seconds.

**Version 2 fixes (after the MIL-101 and CALF-20-Cl tests):**
- *Surface cutoff.* Each atom's energy is multiplied by a weight that is 1 at the surface and 0 deeper than 12 Å, so bulk atoms add nothing. Before this, γ grew with slab thickness: the same ZIF-8 surface gave 0.74, 1.31 and 2.16 J/m² for 30, 60 and 90 Å slabs, and MIL-101's 14,700-atom slabs all came out near −0.45.
- *γ > 0.* The output passes through a softplus, because a Wulff shape needs positive γ on every face.
- *BFDH from the project note.* d_eff = d/n, where n is the first order whose structure factor is not cancelled. It is used as a baseline in cell 11 and printed next to every GNN prediction in cell 13. It reproduces the note's worked numbers: MOF-5 truncated octahedron 77.6 % {111}; Mg-MOF-74 rod, {2̅10} 77.4 %.
- *Out-of-range warning.* Cell 13 lists elements the GNN never saw in training, and flags slabs far larger than any training slab.
- *Shape names* in `mofwulff.name_habit` now follow the note: cube → truncated cube → cuboctahedron (exactly 63.4 % {100}) → truncated octahedron → octahedron, and Zingg 2/3 ratios for rod / plate / lath. Before, a truncated octahedron with more than 20 % {100} was called a cuboctahedron.

**Labels for every termination.** The main run relaxes only the `RELAX_TOP_K = 2` lowest static terminations per face. A GNN trained on those never sees relaxed values for higher-energy terminations, yet it has to rank all of them at inference. For training data, set `RELAX_TOP_K_TRAIN = MAX_TERMS` (cell 9). This costs about 4× more GPU time per MOF.

In [ ]:
%%writefile mofgnn.py
"""GNN surrogate for MOF slab surface energies (option a: structure graph -> gamma).

Teacher : steps 1-6 of mofwulff (healed slab + CHGNet relaxation) -> gamma_relaxed per (MOF, face, termination)
Student : a message-passing GNN on the atom graph of the *unrelaxed* healed slab -> gamma
Then    : lowest predicted gamma per face family -> mofwulff.wulff -> habit (unchanged step 8)

Design rules (each one is a choice, documented here so it can be defended):
  * Input geometry = UNRELAXED healed slab (steps 1-5 only). Relaxation is the expensive step the
    student is meant to skip, so it cannot be an input at inference. Training on relaxed CIFs would
    give a model that needs CHGNet relaxation before it can be used.
  * Edges = the pipeline's own bond rules (mofwulff.bond_kind: org / MM / ML / MO), periodic in-plane.
  * Readout is physical: each atom gets an excess energy e_i (eV); gamma = sum(e_i) / (2A).
    This mirrors gamma = (E_slab - N e_bulk - n E_H2O) / 2A, so the model is size-consistent
    (thicker slab or bigger surface cell -> same gamma).
  * Split by MOF (leave-one-MOF-out, or grouped k-fold for >10 MOFs). A MOF in the test fold never
    appears in training.
  * Baselines are run in every fold: constant mean, linear fit on broken bonds per area, and (when the
    label is relaxed gamma) CHGNet static gamma. The GNN is only useful if it beats them.
  * Surface-localised readout: e_i is multiplied by a smooth cutoff w(depth) that is 1 at the surface
    and 0 deeper than DEPTH_MAX. Atoms deep inside the slab are bulk, so they must add nothing; without
    this, a small per-atom bias adds up in thick slabs (MIL-101: 14,700-atom slabs -> gamma -0.45).
  * positive=True (default for target="gamma") makes gamma = softplus(raw): a crystal with a Wulff shape
    needs gamma > 0 on every face.
  * BFDH (geometry only, project note "CIF to Shape"): d_eff = d_hkl / n, n = first order whose
    structure factor is not cancelled; growth distance proportional to 1/d_eff. Used as a baseline in
    cross-validation and as a cross-check printed next to every GNN prediction.
  * target="delta" learns gamma_relaxed - gamma_static instead (needs one CHGNet single point per slab
    at inference, seconds on a GPU). target="gamma" needs no CHGNet at all.
"""
import json, math, re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pymatgen.core import Element, Lattice, Structure
from scipy.stats import spearmanr

import mofwulff as mw

EV_A2_TO_J_M2 = mw.EV_A2_TO_J_M2
EDGE_KINDS = ["org", "MM", "ML", "MO"]
N_RBF = 16
RBF_MU = np.linspace(0.8, mw.OUT_REACH, N_RBF)
DEPTH_MAX = 12.0                       # A; atoms deeper than this are treated as bulk-like
NODE_FLOAT = 4 + len(EDGE_KINDS) + 2   # is_metal, is_cap, is_donor, is_lig_donor, bond counts, depth, exp(-depth/2)


# ============================================================================ graph building
def slab_graph(st, n_slab_atoms, area_A2):
    """Atom graph of one healed slab.

    Nodes: atomic number (embedded) + float features
           [is_metal, is_cap (atom added by healing), is_donor, is_ligand_donor,
            bond counts per kind (org, MM, ML, MO), depth below nearest surface / DEPTH_MAX, exp(-depth/2)].
    Edges: every pair the pipeline calls a bond, both directions; features = kind one-hot + distance RBF.
    The slab normal is the c axis (block_slabs builds c = n * (thickness + vacuum)); healing appends
    cap atoms after the n_slab_atoms framework atoms, so index >= n_slab_atoms marks a cap atom.
    """
    nbs = st.get_all_neighbors(mw.OUT_REACH)
    lig = mw.ligand_donors(st, nbs)
    sym = [x.specie.symbol for x in st]
    src, dst, kind, dist = [], [], [], []
    counts = np.zeros((len(st), len(EDGE_KINDS)))
    for i in range(len(st)):
        for n in nbs[i]:
            k = mw.bond_kind(sym[i], sym[n.index], n.nn_distance, i in lig, n.index in lig)
            if k:
                src.append(n.index); dst.append(i); kind.append(EDGE_KINDS.index(k)); dist.append(n.nn_distance)
                counts[i, EDGE_KINDS.index(k)] += 1
    chat = st.lattice.matrix[2] / np.linalg.norm(st.lattice.matrix[2])
    z = st.cart_coords @ chat
    frame = z[:n_slab_atoms]
    depth = np.clip(np.minimum(z - frame.min(), frame.max() - z), 0, DEPTH_MAX)
    is_cap = np.arange(len(st)) >= n_slab_atoms
    x = np.c_[[mw.is_metal(s) for s in sym], is_cap, [s in mw.DONORS for s in sym],
              [i in lig for i in range(len(st))], counts, depth / DEPTH_MAX, np.exp(-depth / 2)].astype(np.float32)
    d = np.array(dist)
    ea = np.c_[np.eye(len(EDGE_KINDS))[kind], np.exp(-((d[:, None] - RBF_MU) / 0.15) ** 2)].astype(np.float32)
    return dict(z=torch.tensor([Element(s).Z for s in sym], dtype=torch.long), x=torch.from_numpy(x),
                edge_index=torch.tensor([src, dst], dtype=torch.long), edge_attr=torch.from_numpy(ea),
                area=float(area_A2), n_atoms=len(st))


def read_cif_in_order(path):
    """Read a pipeline CIF with the original site order.

    pymatgen writes sites in order but its CIF parser regroups them by element, so the healing caps
    (appended last) no longer sit at index >= n_slab_atoms. The writer's labels carry the original
    index (O0, H1, C2, ...), so the order is restored from them."""
    st = Structure.from_file(path)
    num = [re.search(r"(\d+)$", str(s.label or "")) for s in st]
    if all(num) and len({int(n.group(1)) for n in num}) == len(st):
        order = np.argsort([int(n.group(1)) for n in num])
        st = Structure.from_sites([st[int(i)] for i in order])
    return st


def _resolve(path, out_dir):
    """slabs.csv stores absolute Colab paths; re-root them under out_dir if the results moved."""
    if isinstance(path, str) and Path(path).exists():
        return path
    if not isinstance(path, str) or "structures/" not in path:
        return None
    p = Path(out_dir) / ("structures/" + path.split("structures/", 1)[1])
    return str(p) if p.exists() else None


def export_graphs(out_dir, slabs_csv=None, cap="water", source="unrelaxed"):
    """Turn every healed slab of slabs.csv into a graph file (out_dir/graphs/*.pt) + index.csv.

    One row per slab with graph path and all labels (gamma_static_J_m2, gamma_relaxed_J_m2, converged).
    source="unrelaxed" (default, see module docstring) or "relaxed" (only for analysis).
    """
    out_dir = Path(out_dir)
    df = pd.read_csv(slabs_csv or out_dir / "slabs.csv")
    for c in ("gamma_relaxed_J_m2", "converged", "unrelaxed_cif", "relaxed_cif"):
        if c not in df:
            df[c] = np.nan                     # static-only runs (QUICK_TEST) have no relaxation columns
    df = df[(df["cap"] == cap) & df["gamma_static_J_m2"].notna()].copy()
    gdir = out_dir / "graphs"; gdir.mkdir(exist_ok=True)
    rows, skipped = [], Counter()
    for _, r in df.iterrows():
        cif = _resolve(r.get("unrelaxed_cif" if source == "unrelaxed" else "relaxed_cif"), out_dir)
        if cif is None:
            skipped["missing cif"] += 1; continue
        p = gdir / (Path(cif).stem + ".pt")
        if not p.exists():
            st = read_cif_in_order(cif)
            n_slab = int(r["n_slab_atoms"])
            if len(st) != int(r["n_atoms"]):
                skipped["atom count mismatch"] += 1; continue
            if not {s.specie.symbol for s in st[n_slab:]} <= {"O", "H"}:
                skipped["cap atoms not O/H (site order lost)"] += 1; continue
            torch.save(slab_graph(st, n_slab, r["area_A2"]), p)
        rows.append({**r.to_dict(), "graph": str(p)})
    idx = pd.DataFrame(rows)
    idx.to_csv(gdir / "index.csv", index=False)
    print(f"{len(idx)} graphs from {idx['mof'].nunique() if len(idx) else 0} MOFs -> {gdir}"
          + (f" | skipped {dict(skipped)}" if skipped else ""))
    return idx


# ============================================================================ model
class GatedConv(nn.Module):
    """CGCNN-style gated message passing with a residual update."""
    def __init__(self, h, e):
        super().__init__()
        self.lin = nn.Linear(2 * h + e, 2 * h)
        self.norm = nn.LayerNorm(h)

    def forward(self, x, ei, ea):
        s, d = ei
        gate, core = self.lin(torch.cat([x[d], x[s], ea], -1)).chunk(2, -1)
        msg = torch.sigmoid(gate) * F.softplus(core)
        agg = torch.zeros_like(x).index_add_(0, d, msg)
        return F.softplus(x + self.norm(agg))


def surface_weight(depth):
    """1 at the surface, 0 deeper than DEPTH_MAX (cosine cutoff)."""
    return 0.5 * (1 + torch.cos(math.pi * torch.clamp(depth, max=DEPTH_MAX) / DEPTH_MAX))


class SlabGNN(nn.Module):
    """Atom graph -> per-atom excess energy (eV) x surface weight -> gamma = sum / 2A (J/m^2)."""
    def __init__(self, hidden=64, layers=4, max_z=100, positive=True):
        super().__init__()
        self.positive = positive
        self.emb = nn.Embedding(max_z + 1, hidden)
        self.inp = nn.Linear(NODE_FLOAT, hidden)
        e_dim = len(EDGE_KINDS) + N_RBF
        self.convs = nn.ModuleList([GatedConv(hidden, e_dim) for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.Softplus(), nn.Linear(hidden, 1))

    def forward(self, b):
        x = self.emb(b["z"]) + self.inp(b["x"])
        for c in self.convs:
            x = c(x, b["edge_index"], b["edge_attr"])
        e_atom = self.head(x).squeeze(-1) * surface_weight(b["x"][:, 8] * DEPTH_MAX)
        e_slab = torch.zeros(b["n_graphs"], device=x.device).index_add_(0, b["batch"], e_atom)
        g = e_slab / (2 * b["area"]) * EV_A2_TO_J_M2
        return F.softplus(g) if self.positive else g      # beta=1: no dead zone for negative raw values


def collate(graphs, device):
    zs, xs, eis, eas, batch, off = [], [], [], [], [], 0
    for i, g in enumerate(graphs):
        zs.append(g["z"]); xs.append(g["x"]); eas.append(g["edge_attr"]); eis.append(g["edge_index"] + off)
        batch.append(torch.full((g["n_atoms"],), i, dtype=torch.long)); off += g["n_atoms"]
    return dict(z=torch.cat(zs).to(device), x=torch.cat(xs).to(device), edge_index=torch.cat(eis, 1).to(device),
                edge_attr=torch.cat(eas).to(device), batch=torch.cat(batch).to(device), n_graphs=len(graphs),
                area=torch.tensor([g["area"] for g in graphs], dtype=torch.float32, device=device))


# ============================================================================ data
def at_water_mu(index, dmu=mw.WATER_DMU_EV["liquid_298K"]):
    """Copy of index.csv with every gamma column moved to water chemical potential dmu (eV per molecule,
    relative to the 0 K molecule used by the pipeline): gamma(dmu) = gamma_0K - n_H2O dmu / 2A.
    The 0 K values are kept as *_0K. Use it before load_dataset / cross_validate so labels, baselines,
    pretraining and the true habits all refer to the same water reference."""
    idx = (index if isinstance(index, pd.DataFrame) else pd.read_csv(index)).copy()
    slope = mw.water_slope(idx["n_water"].fillna(0).to_numpy(), idx["area_A2"].to_numpy())
    for c in ("gamma_static_J_m2", "gamma_relaxed_J_m2"):
        if c in idx:
            idx[c + "_0K"] = idx[c]
            idx[c] = idx[c] - slope * dmu
    idx["water_dmu_eV"] = dmu
    return idx


def load_dataset(index, label="gamma_relaxed_J_m2", target="gamma", require_converged=True, drop_nonpositive=True):
    """Labelled rows of index.csv + their graphs.

    label  : column used as truth. gamma_relaxed_J_m2 for real runs. gamma_static_J_m2 only for
             smoke tests (the pipeline relaxes just RELAX_TOP_K terminations per family).
    target : "gamma" -> model predicts the label; "delta" -> model predicts label - gamma_static.
    Returns (labelled DataFrame, {graph path: graph}) ; unlabelled rows stay in `index` for inference.
    """
    idx = index if isinstance(index, pd.DataFrame) else pd.read_csv(index)
    lab = idx[idx[label].notna()].copy()
    if require_converged and label == "gamma_relaxed_J_m2" and "converged" in lab:
        lab = lab[lab["converged"].astype(str).str.lower() == "true"]
    if drop_nonpositive:
        # gamma <= 0 has no Wulff shape and the GNN output is softplus (> 0): such a label cannot be fitted
        # and says that the water reference is wrong for that slab, not what its surface energy is.
        bad = lab[lab[label] <= 0]
        if len(bad):
            print(f"load_dataset: dropped {len(bad)} slab(s) with {label} <= 0: "
                  + ", ".join(f"{r.mof} {r.face} t{r.termination} ({r[label]:.3f})" for _, r in bad.iterrows()))
        lab = lab[lab[label] > 0]
    lab["y"] = lab[label] - (lab["gamma_static_J_m2"] if target == "delta" else 0.0)
    graphs = {p: torch.load(p, weights_only=False) for p in idx["graph"]}
    return lab.reset_index(drop=True), graphs


def _mof_batches(df, batch_size, rng):
    """Batches that keep a MOF's slabs together, so the ranking loss sees within-MOF pairs."""
    groups = [g.index.to_numpy() for _, g in df.groupby("mof")]
    rng.shuffle(groups)
    cur = []
    for g in groups:
        g = rng.permutation(g)
        for i in range(0, len(g), batch_size):
            cur.extend(g[i:i + batch_size].tolist())
            if len(cur) >= batch_size:
                yield cur; cur = []
    if cur:
        yield cur


def _rank_loss(pred, y, mof_ids, tau=0.1, min_gap=0.02):
    """Pairwise logistic loss on within-MOF orderings (the Wulff shape depends on the ordering)."""
    dy = y[:, None] - y[None, :]
    same = (mof_ids[:, None] == mof_ids[None, :]) & (dy.abs() > min_gap)
    if not same.any():
        return pred.sum() * 0
    dp = pred[:, None] - pred[None, :]
    return F.softplus(-torch.sign(dy[same]) * dp[same] / tau).mean()


# ============================================================================ training
def train(df, graphs, epochs=300, hidden=64, layers=4, lr=1e-3, weight_decay=1e-4, batch_size=16,
          rank_weight=0.5, seed=0, device=None, val_df=None, patience=60, verbose=False, positive=True,
          init_state=None):
    """Train one SlabGNN. If val_df is given, keeps the epoch with the lowest validation MAE.
    init_state: weights to start from (fine-tuning after pretraining)."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SlabGNN(hidden, layers, positive=positive).to(device)
    if init_state is not None:
        model.load_state_dict(init_state)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    mof_code = {m: i for i, m in enumerate(df["mof"].unique())}
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        for ids in _mof_batches(df, batch_size, rng):
            sub = df.loc[ids]
            b = collate([graphs[p] for p in sub["graph"]], device)
            y = torch.tensor(sub["y"].to_numpy(), dtype=torch.float32, device=device)
            m = torch.tensor([mof_code[x] for x in sub["mof"]], device=device)
            pred = model(b)
            loss = F.huber_loss(pred, y, delta=0.3) + rank_weight * _rank_loss(pred, y, m)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        sched.step()
        if val_df is not None and len(val_df):
            mae = float(np.mean(np.abs(predict(model, val_df, graphs, device) - val_df["y"].to_numpy())))
            if mae < best - 1e-4:
                best, bad = mae, 0; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
        if verbose and ep % 50 == 0:
            print(f"epoch {ep} loss {float(loss):.4f}" + (f" val MAE {best:.4f}" if val_df is not None else ""))
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict(model, df, graphs, device=None, batch_size=32):
    device = device or next(model.parameters()).device
    model.eval(); out = []
    paths = list(df["graph"])
    for i in range(0, len(paths), batch_size):
        out.append(model(collate([graphs[p] for p in paths[i:i + batch_size]], device)).cpu().numpy())
    return np.concatenate(out) if out else np.array([])


def train_ensemble(df, graphs, n_seeds=3, val_mofs=0, init_states=None, **kw):
    """n_seeds models; if val_mofs > 0, that many training MOFs are held out for early stopping.
    init_states: one state dict per seed (from pretrain), or None."""
    models = []
    for s in range(n_seeds):
        if init_states is not None:
            kw = {**kw, "init_state": init_states[s % len(init_states)]}
        tr, va = df, None
        if val_mofs:
            mofs = np.random.default_rng(100 + s).permutation(df["mof"].unique())
            va = df[df["mof"].isin(mofs[:val_mofs])]; tr = df[~df["mof"].isin(mofs[:val_mofs])]
        models.append(train(tr.reset_index(drop=True), graphs, seed=s,
                            val_df=None if va is None else va.reset_index(drop=True), **kw))
    return models


def pretrain_states(index, graphs, exclude_mofs=(), n_seeds=3, epochs=100, **kw):
    """Pretrain on CHGNet *static* gamma (cheap: seconds per MOF) of every slab whose MOF is not excluded.
    Returns one state dict per seed, used as the starting point for fine-tuning on relaxed gamma.
    In cross-validation the test MOFs are excluded, so nothing about them is seen before testing."""
    pre = index[index["gamma_static_J_m2"].notna() & (index["gamma_static_J_m2"] > 0)
                & ~index["mof"].isin(list(exclude_mofs))].copy()
    pre["y"] = pre["gamma_static_J_m2"]
    if pre["mof"].nunique() < 2:
        return None
    ms = train_ensemble(pre.reset_index(drop=True), graphs, n_seeds=n_seeds, epochs=epochs,
                        **{k: v for k, v in kw.items() if k not in ("init_state", "epochs")})
    return [{k: v.detach().clone() for k, v in m.state_dict().items()} for m in ms]


def predict_ensemble(models, df, graphs):
    return np.mean([predict(m, df, graphs) for m in models], axis=0)


# ============================================================================ Wulff from any gamma table
def _families_from_meta(meta):
    conv = Structure(Lattice(meta["conv_lattice"]), ["H"], [[0, 0, 0]])
    rots = [np.array(r) for r in meta["_rotations"]]
    fams = mw.face_families(conv, rots, faces=[tuple(f["rep"]) for f in meta["families"]])
    return conv, fams, rots


def habit_from_gammas(meta, gammas):
    """gammas: {family key: gamma}. Returns (habit name, dominant face, wulff dict)."""
    conv, fams, rots = _families_from_meta(meta)
    g = {k: v for k, v in gammas.items() if np.isfinite(v)}
    if len(g) < 2:
        return "not enough faces", None, None
    if min(g.values()) <= 0:
        return "undefined (gamma <= 0)", None, None
    if min(g.values()) < 1e-3:
        return "undefined (gamma ~ 0)", None, None
    try:
        w = mw.wulff(conv, fams, g, rots)
    except Exception as exc:                     # degenerate polyhedron (e.g. qhull on a point)
        return f"undefined (Wulff failed: {type(exc).__name__})", None, None
    if w is None:
        return "no faces", None, None
    name = mw.name_habit(w, fams, meta["crystal_system"], conv)
    names = {f["key"]: f["name"] for f in fams}
    fr = {names.get(k, k): v for k, v in w["area_fractions"].items()}
    return name, max(fr, key=fr.get), w


def habit_class(name):
    """Coarse habit label for agreement checks ('block/prism: {100} 60%, ...' -> 'block/prism')."""
    return re.split(r"[:(]", str(name))[0].strip()


def load_meta(out_dir, mof):
    return json.loads((Path(out_dir) / "cache" / re.sub(r"[^\w.-]", "_", mof) / "meta.json").read_text())


# ============================================================================ BFDH (geometry only)
def structure_factor(st, hkl):
    """|F(hkl)| / sum(Z): Z-weighted sum of exp(2 pi i h.x) over the atoms of st (project note, Step 7)."""
    Z = np.array([x.specie.Z for x in st], float)
    ph = 2 * np.pi * (st.frac_coords @ np.asarray(hkl, float))
    return float(abs(np.sum(Z * np.exp(1j * ph))) / Z.sum())


def _skeleton(info):
    """Atoms used for the cancellation test: the same skeleton that gave the symmetry in step 1.
    H is always left out (ordered H in a CIF breaks symmetry that the framework has)."""
    st = info["conv"].copy()
    els = {x.specie.symbol for x in st}
    if info["symmetry_source"] == "metal+C":
        st.remove_species([e for e in els if not (mw.is_metal(e) or e == "C")])
    elif "H" in els:
        st.remove_species(["H"])
    return st


def d_eff_table(info, families, max_order=8, tol=0.01):
    """d_eff = d_hkl / n for each family; n = first order n*(hkl) whose |F| / sum(Z) > tol.
    Note: rotations make every member of a family equivalent, so the representative is enough."""
    sk = _skeleton(info)
    out = {}
    for f in families:
        h = np.array(f["rep"]); d = float(info["conv"].lattice.d_hkl(tuple(int(x) for x in h)))
        n = next((k for k in range(1, max_order + 1) if structure_factor(sk, k * h) > tol), None)
        out[f["key"]] = dict(name=f["name"], d=round(d, 4), n=n or max_order, d_eff=round(d / (n or max_order), 4),
                             all_orders_cancelled=n is None)
    return out


def bfdh_for_mof(out_dir, mof):
    """d_eff table for a MOF of the pipeline run (cached in cache/<mof>/bfdh.json). None if the CIF is gone."""
    cdir = Path(out_dir) / "cache" / re.sub(r"[^\w.-]", "_", mof)
    if (cdir / "bfdh.json").exists():
        return json.loads((cdir / "bfdh.json").read_text())
    try:
        meta = load_meta(out_dir, mof)
    except FileNotFoundError:
        return None
    if not Path(meta.get("cif", "")).exists():
        return None
    c = meta.get("config", {})
    info = mw.standardize(meta["cif"], c.get("symprec", 0.1), c.get("disorder", "majority"))
    fams = mw.face_families(info["conv"], info["rotations"], faces=[tuple(f["rep"]) for f in meta["families"]])
    t = d_eff_table(info, fams)
    (cdir / "bfdh.json").write_text(json.dumps(t, indent=1))
    return t


def bfdh_habit(meta, table):
    """BFDH shape: Wulff construction with plane distances proportional to 1/d_eff."""
    return habit_from_gammas(meta, {k: 1.0 / v["d_eff"] for k, v in table.items()})


# ============================================================================ saving / loading
def training_elements(lab, graphs):
    return sorted({Element.from_Z(int(z)).symbol for p in lab["graph"] for z in torch.unique(graphs[p]["z"]).tolist()})


def save_models(path, models, lab, graphs, hidden, layers, label, target):
    torch.save(dict(version=2, states=[m.state_dict() for m in models], hidden=hidden, layers=layers,
                    positive=models[0].positive, label=label, target=target,
                    mofs=sorted(lab["mof"].unique()), elements=training_elements(lab, graphs)), path)


def load_models(path, graphs_dir=None, device=None):
    """Returns (models, checkpoint dict). Old checkpoints (before the surface cutoff) are flagged."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(path, weights_only=False, map_location=device)
    if ck.get("version", 1) < 2:
        print("WARNING: this model was trained with the old readout (no surface cutoff, no positivity). "
              "Retrain it (cells 11-12) before trusting predictions.")
    models = []
    for st in ck["states"]:
        m = SlabGNN(ck["hidden"], ck["layers"], positive=ck.get("positive", False)).to(device)
        m.load_state_dict(st); m.eval(); models.append(m)
    if not ck.get("elements") and graphs_dir:
        zs = set()
        for p in Path(graphs_dir).glob("*.pt"):
            zs |= set(torch.load(p, weights_only=False)["z"].tolist())
        ck["elements"] = sorted(Element.from_Z(int(z)).symbol for z in zs)
    return models, ck


# ============================================================================ evaluation
def _family_table(rows, col):
    """Lowest value of `col` per face family."""
    r = rows[rows[col].notna()]
    return r.groupby("family")[col].min().to_dict()


def evaluate_mof(out_dir, mof, test_rows, all_rows, truth_col, pred_cols):
    """Per-MOF metrics for each predictor column.
    truth: lowest labelled gamma per family. prediction: lowest predicted gamma per family over ALL
    terminations (labelled or not), which is how the model is used at inference."""
    meta = load_meta(out_dir, mof)
    true_f = _family_table(test_rows, truth_col)
    true_habit, true_dom, _ = habit_from_gammas(meta, true_f)
    out = []
    for name, col in pred_cols.items():
        if col not in all_rows or all_rows[col].isna().all():
            continue
        pred_f = {k: v for k, v in _family_table(all_rows, col).items() if k in true_f}
        fams = sorted(true_f)
        rho = spearmanr([true_f[k] for k in fams], [pred_f[k] for k in fams]).correlation if len(fams) > 2 else np.nan
        habit, dom, _ = habit_from_gammas(meta, pred_f)
        lab = test_rows[test_rows[col].notna()]
        out.append(dict(mof=mof, model=name, n_labelled=len(lab), n_families=len(fams),
                        MAE_J_m2=float(np.mean(np.abs(lab[col] - lab[truth_col]))),
                        spearman_families=rho,
                        lowest_face_agrees=min(true_f, key=true_f.get) == min(pred_f, key=pred_f.get),
                        true_habit=true_habit, pred_habit=habit,
                        # no Wulff shape exists when the truth has gamma <= 0: not counted as agreement
                        habit_agrees=(habit_class(true_habit) == habit_class(habit)) if true_dom is not None else np.nan,
                        dominant_agrees=(true_dom == dom) if true_dom is not None else np.nan))
    return out


def cross_validate(out_dir, index, label="gamma_relaxed_J_m2", target="gamma", n_folds=None, n_seeds=3,
                   val_mofs=0, pretrain=False, pretrain_epochs=30, pretrain_seeds=1, finetune_lr=3e-4,
                   groups=None, **train_kw):
    """Grouped-by-MOF cross-validation of the GNN against baselines.

    Folds: leave-one-group-out when there are <= 10 groups, otherwise n_folds (default 5) grouped folds.
    groups: {mof: family} so near-copies (Mg/Ni/Zn/Co-MOF-74, the IRMOF series, ...) are always on the
    same side of the split, including during pretraining. MOFs not listed form their own group.
    Returns (per-slab predictions, per-MOF metrics, summary).
    """
    idx = index if isinstance(index, pd.DataFrame) else pd.read_csv(index)
    lab, graphs = load_dataset(idx, label, target)
    mofs = np.array(sorted(lab["mof"].unique()))
    inv_d = {}                                       # (mof, family) -> 1/d_eff, for the BFDH baseline
    for m in idx["mof"].unique():
        t = bfdh_for_mof(out_dir, m)
        if t:
            inv_d.update({(m, k): 1.0 / v["d_eff"] for k, v in t.items()})
    if len(mofs) < 2:
        raise ValueError("need labelled slabs from at least 2 MOFs")
    grp = lambda m: (groups or {}).get(m, m)
    gnames = np.array(sorted({grp(m) for m in mofs}))
    if len(gnames) < 2:
        raise ValueError("need labelled slabs from at least 2 groups")
    k = len(gnames) if (n_folds is None and len(gnames) <= 10) else (n_folds or 5)
    gfolds = np.array_split(np.random.default_rng(0).permutation(gnames), k)
    folds = [np.array([m for m in mofs if grp(m) in set(gf)]) for gf in gfolds]
    all_mofs = idx["mof"].unique()
    preds, metrics = [], []
    for fi, test_mofs in enumerate(folds):
        tr = lab[~lab["mof"].isin(test_mofs)].reset_index(drop=True)
        init = None
        if pretrain:
            test_groups = {grp(m) for m in test_mofs}
            excl = [m for m in all_mofs if grp(m) in test_groups]          # whole test families stay unseen
            init = pretrain_states(idx, graphs, exclude_mofs=excl, n_seeds=pretrain_seeds, epochs=pretrain_epochs,
                                   **{k: v for k, v in train_kw.items() if k in ("hidden", "layers", "batch_size")})
        kw = {"positive": target == "gamma", **train_kw}
        if init is not None:
            kw["lr"] = finetune_lr
        models = train_ensemble(tr, graphs, n_seeds=n_seeds, val_mofs=val_mofs, init_states=init, **kw)
        # baselines fitted on the training MOFs only
        a, b0 = np.polyfit(tr["bonds_per_A2"], tr[label], 1)
        mean_y = tr[label].mean()
        if label != "gamma_static_J_m2":
            s1, s0 = np.polyfit(tr["gamma_static_J_m2"], tr[label], 1)
        trb = tr.assign(inv_d=[inv_d.get((m, f), np.nan) for m, f in zip(tr["mof"], tr["family"])]).dropna(subset=["inv_d"])
        bf = np.polyfit(trb["inv_d"], trb[label], 1) if trb["mof"].nunique() >= 2 else None
        for m in test_mofs:
            all_m = idx[idx["mof"] == m].copy()
            g = predict_ensemble(models, all_m, graphs)
            all_m["pred_gnn"] = g + (all_m["gamma_static_J_m2"].to_numpy() if target == "delta" else 0)
            all_m["pred_mean"] = mean_y
            all_m["pred_bonds_linear"] = a * all_m["bonds_per_A2"] + b0
            if bf is not None:
                iv = np.array([inv_d.get((m, f), np.nan) for f in all_m["family"]])
                if np.isfinite(iv).all():
                    all_m["pred_bfdh_linear"] = bf[0] * iv + bf[1]
            if label != "gamma_static_J_m2":
                all_m["pred_static_chgnet"] = all_m["gamma_static_J_m2"]
                all_m["pred_static_calibrated"] = s1 * all_m["gamma_static_J_m2"] + s0
            all_m["fold"] = fi
            test_rows = all_m[all_m["graph"].isin(lab["graph"])]          # labelled (and converged) slabs
            cols = {c[5:]: c for c in all_m.columns if c.startswith("pred_")}
            metrics += evaluate_mof(out_dir, m, test_rows, all_m, label, cols)
            preds.append(all_m)
        print(f"fold {fi + 1}/{k}: test {list(test_mofs)}, train {tr['mof'].nunique()} MOFs / {len(tr)} slabs"
              + (f", pretrained on static gamma of {len(set(all_mofs) - set(excl))} MOFs" if init is not None else ""))
    P, M = pd.concat(preds), pd.DataFrame(metrics)
    S = M.groupby("model").agg(MAE_J_m2=("MAE_J_m2", "mean"), spearman_families=("spearman_families", "mean"),
                               lowest_face_agrees=("lowest_face_agrees", "mean"), habit_agrees=("habit_agrees", "mean"),
                               dominant_agrees=("dominant_agrees", "mean"), n_mofs=("mof", "count"))
    return P, M, S.sort_values("MAE_J_m2")


# ============================================================================ inference on a new MOF (no CHGNet)
def slabs_only(name, cif, symprec=0.1, disorder="majority", max_index=1, n_families=4, faces=None,
               thickness=15.0, vacuum=15.0, level_tol=0.3, max_terms=8):
    """Steps 1-5 of the pipeline (same rules, same order), no energies. Yields slab rows + structures.
    Note: slabs are cut from the unrelaxed bulk; the training slabs came from the CHGNet-relaxed bulk."""
    info = mw.standardize(cif, symprec, disorder)
    fams = mw.face_families(info["conv"], info["rotations"], max_index, n_families, faces, ops=info["ops"])
    meta = dict(mof=name, conv_lattice=info["conv"].lattice.matrix.tolist(), crystal_system=info["crystal_system"],
                families=[{"key": f["key"], "name": f["name"], "rep": f["rep"]} for f in fams],
                _rotations=[R.tolist() for R in info["rotations"]], formula=info["formula"],
                elements=sorted({x.specie.symbol for x in info["structure"]}))
    meta["bfdh"] = d_eff_table(info, fams)
    rows = []
    for f in fams:
        made = mw.block_slabs(info["conv"], info["prim"], f["rep"], thickness, vacuum, level_tol)
        for m in made:
            m["miss"], m["broken"] = mw.broken_bonds(m)
            m["bond_density"] = sum(v for k, v in m["broken"].items() if k in ("ML", "MO")) / 2 / m["area_A2"]
        made.sort(key=lambda m: m["bond_density"])
        for ti, m in enumerate(made[:max_terms]):
            st, n_w, status = mw.heal(m, m["miss"])
            if st is None:
                continue
            rows.append(dict(mof=name, family=f["key"], face=f["name"], termination=ti, n_slab_atoms=len(m["slab"]),
                             n_atoms=len(st), n_water=n_w, area_A2=m["area_A2"], bonds_per_A2=m["bond_density"],
                             status=status, _st=st))
    return meta, rows


def predict_mof(models, name, cif, target="gamma", train_elements=None, **cfg):
    """CIF -> slabs -> graphs -> GNN gamma per termination -> lowest per family -> Wulff habit.
    Also returns the BFDH shape (geometry only) and the elements the GNN never saw in training."""
    if target == "delta":
        raise ValueError("delta models need CHGNet static gamma; run the pipeline with QUICK_TEST for this MOF")
    meta, rows = slabs_only(name, cif, **cfg)
    graphs = {f"g{i}": slab_graph(r["_st"], r["n_slab_atoms"], r["area_A2"]) for i, r in enumerate(rows)}
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "_st"} for r in rows])
    df["graph"] = list(graphs)
    df["gamma_pred_J_m2"] = predict_ensemble(models, df, graphs)
    fam_g = df.groupby("family")["gamma_pred_J_m2"].min().to_dict()
    habit, dom, w = habit_from_gammas(meta, fam_g)
    b_habit, b_dom, b_w = bfdh_habit(meta, meta["bfdh"])
    unseen = sorted(set(meta["elements"]) - set(train_elements)) if train_elements else []
    warn = []
    if unseen:
        warn.append(f"elements never seen in training: {unseen} -> GNN prediction not reliable")
    if df["n_atoms"].max() > 5000:
        warn.append(f"very large slabs ({df['n_atoms'].max()} atoms); training slabs had < 1000")
    return df, dict(mof=name, habit=habit, dominant_face=dom, gammas=fam_g, bfdh_habit=b_habit,
                    bfdh_dominant_face=b_dom, d_eff={v["name"]: v["d_eff"] for v in meta["bfdh"].values()},
                    unseen_elements=unseen, warnings=warn), w, meta | {"bfdh_wulff": b_w}

## 9. Training data: more MOFs (RASPA2 library, surveyed)

All 105 readable CIFs in the RASPA2 MOF folder (the one cell 3 downloads from) were put through steps 1–5. 81 have at least two faces that can be cut and healed. Two lists come from that survey:

- **`RELAX_LIST` (12 MOFs, full CHGNet relaxation, every kept termination):** chosen for new metals and small slabs. V (MIL-47), Pb (MOF-70), Fe (MIL-88A), Cr (MIL-88B), Co (CoBDP), Ni (Ni-MOF-74), Cd (Cd3BTB2), Zr (MIL-140A), and Zn frameworks of other kinds (MOF-5 = IRMOF-1, IRMOF-15, a pillared paddlewheel, ZnBDC). About 16,700 slab atoms to relax in total, against about 2,600 for Mg-MOF-74. **Time the first two MOFs and extrapolate before running all 12.**
- **`STATIC_LIST` (41 MOFs, static γ only, seconds of CHGNet each):** labels for **pretraining**. The GNN first learns static γ on about 420 slabs, then is fine-tuned on the relaxed ones (cells 11–12). Duplicates of the benchmark MOFs are left out. Slabs over 2,000 atoms or MOFs over 12,000 slab atoms in total are also left out.

Everything is cached, and the CSVs are saved after every MOF, so after a disconnect you just run the cell again. Run cells 1–4 (and the `mofwulff` cell) first. Cell 5 is not needed.

In [ ]:
# 9. Generate GNN training data from the RASPA2 library (cached; run again to continue)
GEN_RELAXED       = True  #@param {type:"boolean"}
GEN_STATIC        = True  #@param {type:"boolean"}
RELAX_TOP_K_TRAIN = 8      #@param {type:"integer"}
RELAX_LIST  = "MOF-1-Dybtsev-dmol, IRMOF-1, IRMOF-15, MIL-140A-DFT, MIL-47, MOF-70, MIL-88A-open, MIL-88B-open, CoBDP, Ni-DOBDC, Cd3BTB2, ZnBDC"  #@param {type:"string"}
STATIC_LIST = "MIL-140B-DFT, MIL-140D-DFT, NU-108-Zn, IRMOF-10, MIL-88A-dry, IRMOF-14, MIL-88C-open, IRMOF-2, IRMOF-4, MIL-140C-DFT, MOF-205, IRMOF-8, MIL-88D-open, Zn-DOBDC, ZnMOF-74, MIL-88B-dry, IRMOF-16, Co-DOBDC, IRMOF-7, IRMOF-6, IRMOF-9, MIL-88C-dry, IRMOF-12, ZnHBDC, Zr-TPDC, MOF-69c, MAF-X8, IRMOF-3, ZIF-90, MIL-88D-dry, ZIF-1, ZIF-77, ZIF-2, MOF-72, IRMOF-13, ZIF-6, ZIF-4, ZIF-9, ZIF-7, ZIF-3, ZIF-5"  #@param {type:"string"}
import os, sys, time
sys.path.insert(0, BASE); import mofwulff as mw
if "eng" not in globals():
    eng = mw.Engine()
base_cfg = dict(symprec=SYMPREC, disorder=DISORDER, max_index=MAX_INDEX, n_families=N_FAMILIES, thickness=THICKNESS,
                vacuum=VACUUM, max_terms=MAX_TERMS, cap="water", fmax=FMAX, water_dmu_eV=WATER_DMU_EV)

def run_list(names, cfg_run, csv_name):
    path_csv = f"{OUT_DIR}/{csv_name}"
    recs_all = pd.read_csv(path_csv).to_dict("records") if os.path.exists(path_csv) else []
    for i, n in enumerate(names):
        try:
            cif = fetch(RASPA + n + ".cif", f"{CIF_DIR}/{n}.cif")
            t = time.time()
            meta, fams, recs, bulk = mw.run_mof(eng, n, cif, OUT_DIR, **cfg_run)
            recs_all = [r for r in recs_all if r["mof"] != n] + recs
            pd.DataFrame(recs_all).to_csv(path_csv, index=False)          # saved after every MOF
            print(f"[{i + 1}/{len(names)}] {n}: {len(recs)} slab rows in {time.time() - t:.0f} s")
        except Exception as e:
            print(f"[{i + 1}/{len(names)}] {n}: FAILED {type(e).__name__}: {e}")

relax_names  = [x.strip() for x in RELAX_LIST.split(",") if x.strip()]
static_names = [x.strip() for x in STATIC_LIST.split(",") if x.strip() and x.strip() not in relax_names]
if GEN_RELAXED:
    run_list(relax_names, {**base_cfg, "relax_steps": RELAX_STEPS, "relax_top_k": RELAX_TOP_K_TRAIN}, "slabs_train.csv")
if GEN_STATIC:
    run_list(static_names, {**base_cfg, "relax_steps": 0, "relax_top_k": 0}, "slabs_static.csv")
deliver("training_data")


In [ ]:
# 10. Export slab graphs (one .pt per healed slab, plus graphs/index.csv)
import importlib, mofgnn as G; importlib.reload(G)
frames = [pd.read_csv(f) for f in (f"{OUT_DIR}/slabs.csv", f"{OUT_DIR}/slabs_train.csv", f"{OUT_DIR}/slabs_static.csv") if os.path.exists(f)]
slabs_all = pd.concat(frames, ignore_index=True)
for c in ("gamma_relaxed_J_m2", "converged"):
    if c not in slabs_all: slabs_all[c] = float("nan")      # static-only (QUICK_TEST) runs
slabs_all["_has_relaxed"] = slabs_all["gamma_relaxed_J_m2"].notna()
slabs_all = (slabs_all.sort_values("_has_relaxed")
             .drop_duplicates(subset=["mof", "family", "termination", "cap"], keep="last").drop(columns="_has_relaxed"))
slabs_all.to_csv(f"{OUT_DIR}/slabs_all.csv", index=False)
INDEX = G.export_graphs(OUT_DIR, f"{OUT_DIR}/slabs_all.csv", cap="water")
conv_ok = INDEX["converged"].astype(str).str.lower().eq("true") & INDEX["gamma_relaxed_J_m2"].notna()
display(INDEX.assign(relaxed_label=conv_ok).groupby("mof").agg(slabs=("graph", "size"), relaxed_labels=("relaxed_label", "sum")))

In [ ]:
# 11. Cross-validation by MOF: GNN vs baselines
LABEL       = "gamma_relaxed_J_m2"  #@param ["gamma_relaxed_J_m2", "gamma_static_J_m2"]
TARGET      = "gamma"               #@param ["gamma", "delta"]
EPOCHS      = 300   #@param {type:"integer"}
N_SEEDS     = 3     #@param {type:"integer"}
HIDDEN      = 64    #@param {type:"integer"}
LAYERS      = 4     #@param {type:"integer"}
RANK_WEIGHT = 0.5   #@param {type:"number"}
EXCLUDE_MOFS = ""   #@param {type:"string"}
PRETRAIN    = True  #@param {type:"boolean"}
PRETRAIN_EPOCHS = 30  #@param {type:"integer"}
# Families: near-copies stay on the same side of every split (and out of pretraining for that fold)
FAMILIES = {"MOF-74": ["Mg-MOF-74", "Ni-DOBDC", "Zn-DOBDC", "Co-DOBDC", "ZnMOF-74"],
            "MIL-88": ["MIL-88A-open", "MIL-88A-dry", "MIL-88B-open", "MIL-88B-dry", "MIL-88C-open", "MIL-88C-dry",
                       "MIL-88D-open", "MIL-88D-dry"],
            "MIL-140": ["MIL-140A-DFT", "MIL-140B-DFT", "MIL-140C-DFT", "MIL-140D-DFT"],
            "IRMOF": ["IRMOF-1", "IRMOF-2", "IRMOF-3", "IRMOF-4", "IRMOF-6", "IRMOF-7", "IRMOF-8", "IRMOF-9", "IRMOF-10",
                      "IRMOF-12", "IRMOF-13", "IRMOF-14", "IRMOF-15", "IRMOF-16"],
            "UiO": ["UiO-66", "Zr-TPDC"], "ZIF-sod": ["ZIF-8", "ZIF-90"], "ZIF-7/9": ["ZIF-7", "ZIF-9"]}
GROUPS = {m: fam for fam, ms in FAMILIES.items() for m in ms}
# EXCLUDE_MOFS: comma-separated MOFs to leave out, e.g. ones whose CHGNet gamma <= 0 is not yet validated
# LABEL = gamma_static_J_m2 is only a plumbing test (static labels, and TARGET must stay "gamma").
excl = {x.strip() for x in EXCLUDE_MOFS.split(",") if x.strip()}
INDEX_USE = G.at_water_mu(INDEX[~INDEX["mof"].isin(excl)].reset_index(drop=True), WATER_DMU_EV)   # labels at the stated water condition
lab_mofs = INDEX_USE[INDEX_USE[LABEL].notna()]["mof"].nunique()
if lab_mofs < 2:
    print(f"Only {lab_mofs} MOF(s) have {LABEL}. Run steps 1-6 with relaxation (QUICK_TEST off) and/or cell 9 first.")
else:
    TRAIN_KW = dict(epochs=EPOCHS, hidden=HIDDEN, layers=LAYERS, rank_weight=RANK_WEIGHT)
    P, M, S = G.cross_validate(OUT_DIR, INDEX_USE, label=LABEL, target=TARGET, n_seeds=N_SEEDS,
                               val_mofs=1 if lab_mofs >= 8 else 0, pretrain=PRETRAIN and LABEL != "gamma_static_J_m2",
                               pretrain_epochs=PRETRAIN_EPOCHS, groups=GROUPS, **TRAIN_KW)
    P.to_csv(f"{OUT_DIR}/gnn_cv_predictions.csv", index=False)
    M.to_csv(f"{OUT_DIR}/gnn_cv_per_mof.csv", index=False); S.to_csv(f"{OUT_DIR}/gnn_cv_summary.csv")
    print("Averages over held-out MOFs (higher is better except MAE):"); display(S.round(3))
    display(M[["mof", "model", "n_labelled", "MAE_J_m2", "spearman_families", "lowest_face_agrees",
               "true_habit", "pred_habit", "habit_agrees"]].round(3))

In [ ]:
# 12. Train the final model on all labelled MOFs and save it
if lab_mofs >= 2:
    lab, GRAPHS = G.load_dataset(INDEX_USE, LABEL, TARGET)
    INIT = None
    if PRETRAIN and LABEL != "gamma_static_J_m2":
        INIT = G.pretrain_states(INDEX_USE, GRAPHS, n_seeds=1, epochs=PRETRAIN_EPOCHS, hidden=HIDDEN, layers=LAYERS)
    FINAL = G.train_ensemble(lab, GRAPHS, n_seeds=N_SEEDS, positive=TARGET == "gamma", init_states=INIT,
                             **{**TRAIN_KW, **({"lr": 3e-4} if INIT else {})})
    G.save_models(f"{OUT_DIR}/gnn_surrogate.pt", FINAL, lab, GRAPHS, HIDDEN, LAYERS, LABEL, TARGET)
    print(f"saved {OUT_DIR}/gnn_surrogate.pt  ({len(lab)} slabs, {lab['mof'].nunique()} MOFs, "
          f"elements {G.training_elements(lab, GRAPHS)})")
deliver("final")


In [ ]:
# 13. Predict the shape of a new MOF from its CIF: steps 1-5 + GNN (no CHGNet), with BFDH next to it
# Works in a fresh session: needs only cells 1, 2 (same SAVE_TO_DRIVE), the mofwulff cell and the mofgnn cell.
PREDICT_NEW_MOF = False  #@param {type:"boolean"}
NEW_CIF = ""  #@param {type:"string"}
# Set PREDICT_NEW_MOF and put the CIF file name in NEW_CIF (in the working folder); on Colab, leave NEW_CIF empty to upload.
if PREDICT_NEW_MOF:
    import sys, importlib; sys.path.insert(0, BASE)
    import mofwulff as mw, mofgnn as G; importlib.reload(mw); importlib.reload(G)
    from IPython.display import Image, display
    MODELS, CK = G.load_models(f"{OUT_DIR}/gnn_surrogate.pt", graphs_dir=f"{OUT_DIR}/graphs")
    print("GNN trained on:", CK["mofs"], "| elements:", CK.get("elements"))
    if NEW_CIF:
        fname = NEW_CIF
    else:
        from google.colab import files
        fname = list(files.upload())[0]
    cif = os.path.join(BASE, fname); name = fname.rsplit(".", 1)[0]
    df_new, summ, w_new, meta_new = G.predict_mof(MODELS, name, cif, target=CK["target"], train_elements=CK.get("elements"),
                                                  n_families=4)
    display(df_new[["face", "termination", "n_atoms", "n_water", "bonds_per_A2", "gamma_pred_J_m2"]].round(4))
    for wmsg in summ["warnings"]:
        print("WARNING:", wmsg)
    print("GNN shape :", summ["habit"], "| biggest face:", summ["dominant_face"])
    print("BFDH shape:", summ["bfdh_habit"], "| biggest face:", summ["bfdh_dominant_face"], "| d_eff (A):", summ["d_eff"])
    fams_new = G._families_from_meta(meta_new)[1]
    for label, wv, hab in [("GNN", w_new, summ["habit"]), ("BFDH", meta_new["bfdh_wulff"], summ["bfdh_habit"])]:
        if wv is not None:
            display(Image(mw.render(wv, fams_new, f"{name} ({label})\n{hab}", os.path.join(OUT_DIR, f"wulff_{name}_{label}.png"))))


## What to report, and the limits

- **Main table (cell 11):** for each held-out MOF, the error in γ (MAE), the rank correlation across face families, whether the lowest-energy face matches, and whether the Wulff habit matches. Show the GNN next to every baseline. If `bonds_linear` or `static_chgnet` wins, that is the result, and it is worth reporting.
- **Teacher, not truth:** the GNN learns CHGNet's γ. Its errors against real surfaces are at least CHGNet's errors, and those are still unvalidated until step 7c (DFT) is done.
- **Scale:** with fewer than about 20 MOFs, a single new chemistry (e.g. the only Zr MOF) is an extrapolation. A 4-MOF smoke test gave GNN MAE of 3.3 J/m² on UiO-66 when no Zr MOF was in training.
- **Inference slabs** in cell 13 are cut from the unrelaxed bulk; training slabs come from the CHGNet-relaxed bulk. The bonds are the same; distances differ slightly.
- **CIF site order:** pymatgen's CIF reader regroups atoms by element. `G.read_cif_in_order` restores the order from the site labels, so the water caps are still recognised.

In [ ]:
# 13. Predict the shape of a new MOF from its CIF: steps 1-5 + GNN (no CHGNet), with BFDH next to it
# Works in a fresh session: needs only cells 1, 2 (same SAVE_TO_DRIVE), the mofwulff cell and the mofgnn cell.
PREDICT_NEW_MOF = False  #@param {type:"boolean"}
NEW_CIF = ""  #@param {type:"string"}
# Set PREDICT_NEW_MOF and put the CIF file name in NEW_CIF (in the working folder); on Colab, leave NEW_CIF empty to upload.
if PREDICT_NEW_MOF:
    import sys, importlib; sys.path.insert(0, BASE)
    import mofwulff as mw, mofgnn as G; importlib.reload(mw); importlib.reload(G)
    from IPython.display import Image, display
    MODELS, CK = G.load_models(f"{OUT_DIR}/gnn_surrogate.pt", graphs_dir=f"{OUT_DIR}/graphs")
    print("GNN trained on:", CK["mofs"], "| elements:", CK.get("elements"))
    if NEW_CIF:
        fname = NEW_CIF
    else:
        from google.colab import files
        fname = list(files.upload())[0]
    cif = os.path.join(BASE, fname); name = fname.rsplit(".", 1)[0]
    df_new, summ, w_new, meta_new = G.predict_mof(MODELS, name, cif, target=CK["target"], train_elements=CK.get("elements"),
                                                  n_families=4)
    display(df_new[["face", "termination", "n_atoms", "n_water", "bonds_per_A2", "gamma_pred_J_m2"]].round(4))
    for wmsg in summ["warnings"]:
        print("WARNING:", wmsg)
    print("GNN shape :", summ["habit"], "| biggest face:", summ["dominant_face"])
    print("BFDH shape:", summ["bfdh_habit"], "| biggest face:", summ["bfdh_dominant_face"], "| d_eff (A):", summ["d_eff"])
    fams_new = G._families_from_meta(meta_new)[1]
    for label, wv, hab in [("GNN", w_new, summ["habit"]), ("BFDH", meta_new["bfdh_wulff"], summ["bfdh_habit"])]:
        if wv is not None:
            display(Image(mw.render(wv, fams_new, f"{name} ({label})\n{hab}", os.path.join(OUT_DIR, f"wulff_{name}_{label}.png"))))
